In [ ]:
# install gurobipy pip install gurobipy  



In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from IPython.display import Image
import io
from gurobipy import quicksum


# Problem Formulation


## Importing Data

In [ ]:
Index = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\index.csv')

## Sets

In [ ]:
# Warehouses
#W= list(Warehouse_data['Warehouse'])

# Years
Y = list(Index['Year'])
Y = [x for x in Y if str(x) != 'nan']

Y_warm = list(range(2025,2101))
Y_start = [2020, 2021, 2022, 2023, 2024]

# Stage
S = list(Index['Stage'])
S = [x for x in S if str(x) != 'nan']

# # Region
# R = list(Index['Region'])
# R = [x for x in R if str(x) != 'nan']
R = ['US','China']
# Technology
T = list(Index['Technology'])
T = [x for x in T if str(x) != 'nan']

#cost adjustment coefficient
alpha = {'US': 1.3, 'China': 1}

#Capacity adjustment coefficient
beta= {'US': 1, 'China': 1}

#MarketShare adjustment coeff
gamma= {'US': 0.5, 'China': 0.5}

#import and export cost coefficient
imp={'US': 1.1, 'China': 1.3}

invR=0.01

impCO2=1

print(R)

## Parameters

### Demand

In [ ]:
Mine_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_demand.csv')

Proc_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\proc_demand.csv')

Cath_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cath_demand.csv')

Cell_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cell_demand.csv')

Pack_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\pack_demand.csv')

Rec_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\rec_demand.csv')

In [ ]:
total_Pack_Demand = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\pack_demand_total.csv')

In [ ]:
# Demand

YSR_mine, mine_demand = gp.multidict((Mine_Demand.set_index(zip(Mine_Demand['Year'], Mine_Demand['Stage'], Mine_Demand['Region'])))['Demand'].to_dict())

YSTR_proc, proc_demand = gp.multidict((Proc_Demand.set_index(zip(Proc_Demand['Year'], Proc_Demand['Stage'], Proc_Demand['Technology'], Proc_Demand['Region'])))['Demand'].to_dict())

YSTR_cath, cath_demand = gp.multidict((Cath_Demand.set_index(zip(Cath_Demand['Year'], Cath_Demand['Stage'], Cath_Demand['Technology'], Cath_Demand['Region'])))['Demand'].to_dict())

YSTR_cell, cell_demand = gp.multidict((Cell_Demand.set_index(zip(Cell_Demand['Year'], Cell_Demand['Stage'], Cell_Demand['Technology'] , Cell_Demand['Region'])))['Demand'].to_dict())

YSTR_pack, pack_demand = gp.multidict((Pack_Demand.set_index(zip(Pack_Demand['Year'], Pack_Demand['Stage'], Pack_Demand['Technology'], Pack_Demand['Region'])))['Demand'].to_dict())

YSTR_rec, rec_demand = gp.multidict((Rec_Demand.set_index(zip(Rec_Demand['Year'], Rec_Demand['Stage'], Rec_Demand['Technology'], Rec_Demand['Region'])))['Demand'].to_dict())

In [ ]:
YST_pack, pack_demand_total = gp.multidict((total_Pack_Demand.set_index(zip(total_Pack_Demand['Year'], total_Pack_Demand['Stage'], total_Pack_Demand['Technology'])))['Demand'].to_dict())

### Mine Cost

In [ ]:
Mine_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_prod_cost.csv')

Mine_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_var_cost.csv')

Mine_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_fixed_cost.csv')


Mine_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_import_cost.csv')
Mine_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_export_cost.csv')
Mine_Stock_Cost=pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_stock_cost.csv')

In [ ]:
Mine_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_prod_CO2.csv')

Mine_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_var_CO2.csv')

Mine_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_fixed_CO2.csv')

Mine_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_import_co2.csv')

Mine_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\mine_export_co2.csv')

In [ ]:
# Mine Costs

YSTR_mine, mine_prod_cost = gp.multidict((Mine_Prod_Cost.set_index(zip(Mine_Prod_Cost['Year'], Mine_Prod_Cost['Stage'], Mine_Prod_Cost['Technology'], Mine_Prod_Cost['Region'])))['Cost'].to_dict())

YSTR_mine, mine_fixed_cost = gp.multidict((Mine_Fixed_Cost.set_index(zip(Mine_Fixed_Cost['Year'], Mine_Fixed_Cost['Stage'], Mine_Fixed_Cost['Technology'], Mine_Fixed_Cost['Region'])))['Cost'].to_dict())

YSTR_mine, mine_var_cost = gp.multidict((Mine_Var_Cost.set_index(zip(Mine_Var_Cost['Year'], Mine_Var_Cost['Stage'], Mine_Var_Cost['Technology'], Mine_Var_Cost['Region'])))['Cost'].to_dict())

YSTRR_mine, mine_imp_cost = gp.multidict((Mine_Imp_Cost.set_index(zip(Mine_Imp_Cost['Year'], Mine_Imp_Cost['Stage'], Mine_Imp_Cost['Technology'], Mine_Imp_Cost['RegionI'], Mine_Imp_Cost['RegionO']  )))['Cost'].to_dict())

YSTRR_mine, mine_exp_cost = gp.multidict((Mine_Exp_Cost.set_index(zip(Mine_Exp_Cost['Year'], Mine_Exp_Cost['Stage'], Mine_Exp_Cost['Technology'], Mine_Exp_Cost['RegionI'], Mine_Exp_Cost['RegionO']  )))['Cost'].to_dict())

YSTR_mine, mine_stock_cost = gp.multidict((Mine_Stock_Cost.set_index(zip(Mine_Stock_Cost['Year'], Mine_Stock_Cost['Stage'], Mine_Stock_Cost['Technology'], Mine_Stock_Cost['Region'])))['Cost'].to_dict())


In [ ]:
# Mine CO2s

YSTR_mine, mine_prod_CO2 = gp.multidict((Mine_Prod_CO2.set_index(zip(Mine_Prod_CO2['Year'], Mine_Prod_CO2['Stage'], Mine_Prod_CO2['Technology'], Mine_Prod_CO2['Region'])))['CO2'].to_dict())

YSTR_mine, mine_fixed_CO2 = gp.multidict((Mine_Fixed_CO2.set_index(zip(Mine_Fixed_CO2['Year'], Mine_Fixed_CO2['Stage'], Mine_Fixed_CO2['Technology'], Mine_Fixed_CO2['Region'])))['CO2'].to_dict())

YSTR_mine, mine_var_CO2 = gp.multidict((Mine_Var_CO2.set_index(zip(Mine_Var_CO2['Year'], Mine_Var_CO2['Stage'], Mine_Var_CO2['Technology'], Mine_Var_CO2['Region'])))['CO2'].to_dict())

YSTRR_mine, mine_imp_CO2 = gp.multidict((Mine_Imp_CO2.set_index(zip(Mine_Imp_CO2['Year'], Mine_Imp_CO2['Stage'], Mine_Imp_CO2['Technology'], Mine_Imp_CO2['RegionI'], Mine_Imp_CO2['RegionO']  )))['CO2'].to_dict())

YSTRR_mine, mine_exp_CO2 = gp.multidict((Mine_Exp_CO2.set_index(zip(Mine_Exp_CO2['Year'], Mine_Exp_CO2['Stage'], Mine_Exp_CO2['Technology'], Mine_Exp_CO2['RegionI'], Mine_Exp_CO2['RegionO']  )))['CO2'].to_dict())


### Processing Cost

In [ ]:
Proc_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_prod_cost.csv')

Proc_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_var_cost.csv')

Proc_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_fixed_cost.csv')

Proc_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_import_cost.csv')

Proc_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_export_cost.csv')

In [ ]:
Proc_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_prod_CO2.csv')

Proc_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_var_CO2.csv')

Proc_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_fixed_CO2.csv')

Proc_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_import_CO2.csv')

Proc_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Proc_export_CO2.csv')

In [ ]:
YSIOR_proc, proc_prod_cost = gp.multidict((Proc_Prod_Cost.set_index(zip(Proc_Prod_Cost['Year'], Proc_Prod_Cost['Stage'], Proc_Prod_Cost['Input'], Proc_Prod_Cost['Output'], Proc_Prod_Cost['Region'])))['Cost'].to_dict())

YSIOR_proc, proc_fixed_cost = gp.multidict((Proc_Fixed_Cost.set_index(zip(Proc_Fixed_Cost['Year'], Proc_Fixed_Cost['Stage'], Proc_Fixed_Cost['Input'], Proc_Fixed_Cost['Output'], Proc_Fixed_Cost['Region'])))['Cost'].to_dict())

YSIOR_proc, proc_var_cost = gp.multidict((Proc_Var_Cost.set_index(zip(Proc_Var_Cost['Year'], Proc_Var_Cost['Stage'], Proc_Var_Cost['Input'], Proc_Var_Cost['Output'], Proc_Var_Cost['Region'])))['Cost'].to_dict())

YSIORR_proc, proc_imp_cost = gp.multidict((Proc_Imp_Cost.set_index(zip(Proc_Imp_Cost['Year'], Proc_Imp_Cost['Stage'], Proc_Imp_Cost['Input'], Proc_Imp_Cost['Output'], Proc_Imp_Cost['RegionI'], Proc_Imp_Cost['RegionO'])))['Cost'].to_dict())

YSIORR_proc, proc_exp_cost = gp.multidict((Proc_Exp_Cost.set_index(zip(Proc_Exp_Cost['Year'], Proc_Exp_Cost['Stage'], Proc_Exp_Cost['Input'], Proc_Exp_Cost['Output'], Proc_Exp_Cost['RegionI'], Proc_Exp_Cost['RegionO'])))['Cost'].to_dict())

In [ ]:
YSIOR_proc, proc_prod_CO2 = gp.multidict((Proc_Prod_CO2.set_index(zip(Proc_Prod_CO2['Year'], Proc_Prod_CO2['Stage'], Proc_Prod_CO2['Input'], Proc_Prod_CO2['Output'], Proc_Prod_CO2['Region'] )))['CO2'].to_dict())

YSIOR_proc, proc_fixed_CO2 = gp.multidict((Proc_Fixed_CO2.set_index(zip(Proc_Fixed_CO2['Year'], Proc_Fixed_CO2['Stage'], Proc_Fixed_CO2['Input'], Proc_Fixed_CO2['Output'], Proc_Fixed_CO2['Region'])))['CO2'].to_dict())

YSIOR_proc, proc_var_CO2 = gp.multidict((Proc_Var_CO2.set_index(zip(Proc_Var_CO2['Year'], Proc_Var_CO2['Stage'], Proc_Var_CO2['Input'], Proc_Var_CO2['Output'], Proc_Var_CO2['Region'])))['CO2'].to_dict())

YSIORR_proc, proc_imp_CO2 = gp.multidict((Proc_Imp_CO2.set_index(zip(Proc_Imp_CO2['Year'], Proc_Imp_CO2['Stage'], Proc_Imp_CO2['Input'], Proc_Imp_CO2['Output'], Proc_Imp_CO2['RegionI'], Proc_Imp_CO2['RegionO'])))['CO2'].to_dict())

YSIORR_proc, proc_exp_CO2 = gp.multidict((Proc_Exp_CO2.set_index(zip(Proc_Exp_CO2['Year'], Proc_Exp_CO2['Stage'], Proc_Exp_CO2['Input'], Proc_Exp_CO2['Output'], Proc_Exp_CO2['RegionI'], Proc_Exp_CO2['RegionO'])))['CO2'].to_dict())

### Cathode Cost

In [ ]:
Cath_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_prod_cost.csv')

Cath_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_var_cost.csv')

Cath_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_fixed_cost.csv')

Cath_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_import_cost.csv')

Cath_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_export_cost.csv')

In [ ]:
Cath_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cath_prod_CO2.csv')

Cath_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cath_var_CO2.csv')

Cath_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cath_fixed_CO2.csv')

Cath_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_import_CO2.csv')

Cath_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cath_export_CO2.csv')

In [ ]:
YSIOR_cath, cath_prod_cost = gp.multidict((Cath_Prod_Cost.set_index(zip(Cath_Prod_Cost['Year'], Cath_Prod_Cost['Stage'], Cath_Prod_Cost['Input'], Cath_Prod_Cost['Output'], Cath_Prod_Cost['Region'] )))['Cost'].to_dict())

YSIOR_cath, cath_fixed_cost = gp.multidict((Cath_Fixed_Cost.set_index(zip(Cath_Fixed_Cost['Year'], Cath_Fixed_Cost['Stage'], Cath_Fixed_Cost['Input'], Cath_Fixed_Cost['Output'], Cath_Fixed_Cost['Region'])))['Cost'].to_dict())

YSIOR_cath, cath_var_cost = gp.multidict((Cath_Var_Cost.set_index(zip(Cath_Var_Cost['Year'], Cath_Var_Cost['Stage'], Cath_Var_Cost['Input'], Cath_Var_Cost['Output'], Cath_Var_Cost['Region'])))['Cost'].to_dict())

YSIORR_cath, cath_imp_cost = gp.multidict((Cath_Imp_Cost.set_index(zip(Cath_Imp_Cost['Year'], Cath_Imp_Cost['Stage'], Cath_Imp_Cost['Input'], Cath_Imp_Cost['Output'], Cath_Imp_Cost['RegionI'], Cath_Imp_Cost['RegionO'])))['Cost'].to_dict())

YSIORR_cath, cath_exp_cost = gp.multidict((Cath_Exp_Cost.set_index(zip(Cath_Exp_Cost['Year'], Cath_Exp_Cost['Stage'], Cath_Exp_Cost['Input'], Cath_Exp_Cost['Output'], Cath_Exp_Cost['RegionI'], Cath_Exp_Cost['RegionO'])))['Cost'].to_dict())

In [ ]:
YSIOR_cath, cath_prod_CO2 = gp.multidict((Cath_Prod_CO2.set_index(zip(Cath_Prod_CO2['Year'], Cath_Prod_CO2['Stage'], Cath_Prod_CO2['Input'], Cath_Prod_CO2['Output'], Cath_Prod_CO2['Region'] )))['CO2'].to_dict())

YSIOR_cath, cath_fixed_CO2 = gp.multidict((Cath_Fixed_CO2.set_index(zip(Cath_Fixed_CO2['Year'], Cath_Fixed_CO2['Stage'], Cath_Fixed_CO2['Input'], Cath_Fixed_CO2['Output'], Cath_Fixed_CO2['Region'])))['CO2'].to_dict())

YSIOR_cath, cath_var_CO2 = gp.multidict((Cath_Var_CO2.set_index(zip(Cath_Var_CO2['Year'], Cath_Var_CO2['Stage'], Cath_Var_CO2['Input'], Cath_Var_CO2['Output'], Cath_Var_CO2['Region'])))['CO2'].to_dict())

YSIORR_cath, cath_imp_CO2 = gp.multidict((Cath_Imp_CO2.set_index(zip(Cath_Imp_CO2['Year'], Cath_Imp_CO2['Stage'], Cath_Imp_CO2['Input'], Cath_Imp_CO2['Output'], Cath_Imp_CO2['RegionI'], Cath_Imp_Cost['RegionO'])))['CO2'].to_dict())

YSIORR_cath, cath_exp_CO2 = gp.multidict((Cath_Exp_CO2.set_index(zip(Cath_Exp_CO2['Year'], Cath_Exp_CO2['Stage'], Cath_Exp_CO2['Input'], Cath_Exp_CO2['Output'], Cath_Exp_CO2['RegionI'], Cath_Exp_Cost['RegionO'])))['CO2'].to_dict())

### Cell Cost (Added Import Export Cost for Cell do same for all stages and for CO2 as well)

In [ ]:
Cell_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_prod_cost.csv')

Cell_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_var_cost.csv')

Cell_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_fixed_cost.csv')

## Import Export, need one for import and export and stockpile
Cell_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cell_import_cost.csv')

Cell_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\cell_export_cost.csv')

In [ ]:
Cell_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_prod_CO2.csv')

Cell_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_var_CO2.csv')

Cell_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_fixed_CO2.csv')

Cell_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_import_CO2.csv')

Cell_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Cell_export_CO2.csv')

In [ ]:
YSIOR_cell, cell_prod_cost = gp.multidict((Cell_Prod_Cost.set_index(zip(Cell_Prod_Cost['Year'], Cell_Prod_Cost['Stage'], Cell_Prod_Cost['Input'], Cell_Prod_Cost['Output'], Cell_Prod_Cost['Region']  )))['Cost'].to_dict())

YSIOR_cell, cell_fixed_cost = gp.multidict((Cell_Fixed_Cost.set_index(zip(Cell_Fixed_Cost['Year'], Cell_Fixed_Cost['Stage'], Cell_Fixed_Cost['Input'], Cell_Fixed_Cost['Output'], Cell_Fixed_Cost['Region'])))['Cost'].to_dict())

YSIOR_cell, cell_var_cost = gp.multidict((Cell_Var_Cost.set_index(zip(Cell_Var_Cost['Year'], Cell_Var_Cost['Stage'], Cell_Var_Cost['Input'], Cell_Var_Cost['Output'], Cell_Var_Cost['Region'])))['Cost'].to_dict())

## Import Export New Index for Region Imported to and Region Exported From
YSIORR_cell, cell_import_cost = gp.multidict((Cell_Imp_Cost.set_index(zip(Cell_Imp_Cost['Year'], Cell_Imp_Cost['Stage'], Cell_Imp_Cost['Input'], Cell_Imp_Cost['Output'], Cell_Imp_Cost['RegionI'], Cell_Imp_Cost['RegionO']  )))['Cost'].to_dict())

YSIORR_cell, cell_export_cost = gp.multidict((Cell_Exp_Cost.set_index(zip(Cell_Exp_Cost['Year'], Cell_Exp_Cost['Stage'], Cell_Exp_Cost['Input'], Cell_Exp_Cost['Output'], Cell_Exp_Cost['RegionI'], Cell_Exp_Cost['RegionO']  )))['Cost'].to_dict())

In [ ]:
YSIOR_cell, cell_prod_CO2 = gp.multidict((Cell_Prod_CO2.set_index(zip(Cell_Prod_CO2['Year'], Cell_Prod_CO2['Stage'], Cell_Prod_CO2['Input'], Cell_Prod_CO2['Output'], Cell_Prod_CO2['Region'] )))['CO2'].to_dict())

YSIOR_cell, cell_fixed_CO2 = gp.multidict((Cell_Fixed_CO2.set_index(zip(Cell_Fixed_CO2['Year'], Cell_Fixed_CO2['Stage'], Cell_Fixed_CO2['Input'], Cell_Fixed_CO2['Output'], Cell_Fixed_CO2['Region'])))['CO2'].to_dict())

YSIOR_cell, cell_var_CO2 = gp.multidict((Cell_Var_CO2.set_index(zip(Cell_Var_CO2['Year'], Cell_Var_CO2['Stage'], Cell_Var_CO2['Input'], Cell_Var_CO2['Output'], Cell_Var_CO2['Region'])))['CO2'].to_dict())

YSIORR_cell, cell_import_CO2 = gp.multidict((Cell_Imp_CO2.set_index(zip(Cell_Imp_CO2['Year'], Cell_Imp_CO2['Stage'], Cell_Imp_CO2['Input'], Cell_Imp_CO2['Output'], Cell_Imp_CO2['RegionI'], Cell_Imp_CO2['RegionO']  )))['CO2'].to_dict())

YSIORR_cell, cell_export_CO2 = gp.multidict((Cell_Exp_CO2.set_index(zip(Cell_Exp_CO2['Year'], Cell_Exp_CO2['Stage'], Cell_Exp_CO2['Input'], Cell_Exp_Cost['Output'], Cell_Exp_Cost['RegionI'], Cell_Exp_Cost['RegionO']  )))['CO2'].to_dict())

### Pack Cost

In [ ]:
Pack_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_prod_cost.csv')

Pack_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_var_cost.csv')

Pack_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_fixed_cost.csv')

Pack_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_import_cost.csv')

Pack_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_export_cost.csv')

In [ ]:
Pack_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_prod_CO2.csv')

Pack_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_var_CO2.csv')

Pack_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_fixed_CO2.csv')

Pack_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_import_CO2.csv')

Pack_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Pack_export_CO2.csv')

In [ ]:
YSIOR_pack, pack_prod_cost = gp.multidict((Pack_Prod_Cost.set_index(zip(Pack_Prod_Cost['Year'], Pack_Prod_Cost['Stage'], Pack_Prod_Cost['Input'], Pack_Prod_Cost['Output'], Pack_Prod_Cost['Region'] )))['Cost'].to_dict())

YSIOR_pack, pack_fixed_cost = gp.multidict((Pack_Fixed_Cost.set_index(zip(Pack_Fixed_Cost['Year'], Pack_Fixed_Cost['Stage'], Pack_Fixed_Cost['Input'], Pack_Fixed_Cost['Output'], Pack_Fixed_Cost['Region'])))['Cost'].to_dict())

YSIOR_pack, pack_var_cost = gp.multidict((Pack_Var_Cost.set_index(zip(Pack_Var_Cost['Year'], Pack_Var_Cost['Stage'], Pack_Var_Cost['Input'], Pack_Var_Cost['Output'], Pack_Var_Cost['Region'])))['Cost'].to_dict())

YSIORR_pack, pack_import_cost = gp.multidict((Pack_Imp_Cost.set_index(zip(Pack_Imp_Cost['Year'], Pack_Imp_Cost['Stage'], Pack_Imp_Cost['Input'], Pack_Imp_Cost['Output'], Pack_Imp_Cost['RegionI'], Pack_Imp_Cost['RegionO']  )))['Cost'].to_dict())

YSIORR_pack, pack_export_cost = gp.multidict((Pack_Exp_Cost.set_index(zip(Pack_Exp_Cost['Year'], Pack_Exp_Cost['Stage'], Pack_Exp_Cost['Input'], Pack_Exp_Cost['Output'], Pack_Exp_Cost['RegionI'], Pack_Exp_Cost['RegionO']  )))['Cost'].to_dict())

In [ ]:
YSIOR_pack, pack_prod_CO2 = gp.multidict((Pack_Prod_CO2.set_index(zip(Pack_Prod_CO2['Year'], Pack_Prod_CO2['Stage'], Pack_Prod_CO2['Input'], Pack_Prod_CO2['Output'], Pack_Prod_CO2['Region'] )))['CO2'].to_dict())

YSIOR_pack, pack_fixed_CO2 = gp.multidict((Pack_Fixed_CO2.set_index(zip(Pack_Fixed_CO2['Year'], Pack_Fixed_CO2['Stage'], Pack_Fixed_CO2['Input'], Pack_Fixed_CO2['Output'], Pack_Fixed_CO2['Region'])))['CO2'].to_dict())

YSIOR_pack, pack_var_CO2 = gp.multidict((Pack_Var_CO2.set_index(zip(Pack_Var_CO2['Year'], Pack_Var_CO2['Stage'], Pack_Var_CO2['Input'], Pack_Var_CO2['Output'], Pack_Var_CO2['Region'])))['CO2'].to_dict())

YSIORR_pack, pack_import_CO2 = gp.multidict((Pack_Imp_CO2.set_index(zip(Pack_Imp_Cost['Year'], Pack_Imp_CO2['Stage'], Pack_Imp_CO2['Input'], Pack_Imp_CO2['Output'], Pack_Imp_CO2['RegionI'], Pack_Imp_CO2['RegionO']  )))['CO2'].to_dict())

YSIORR_pack, pack_export_CO2 = gp.multidict((Pack_Exp_CO2.set_index(zip(Pack_Exp_Cost['Year'], Pack_Exp_CO2['Stage'], Pack_Exp_CO2['Input'], Pack_Exp_CO2['Output'], Pack_Exp_CO2['RegionI'], Pack_Exp_CO2['RegionO']  )))['CO2'].to_dict())

### Recycle Cost

In [ ]:
Rec_Prod_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_prod_cost.csv')

Rec_Var_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_var_cost.csv')

Rec_Fixed_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_fixed_cost.csv')

Rec_Imp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_import_cost.csv')

Rec_Exp_Cost = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_export_cost.csv')

In [ ]:
Rec_Prod_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_prod_CO2.csv')

Rec_Var_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_var_CO2.csv')

Rec_Fixed_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_fixed_CO2.csv')

Rec_Imp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_import_CO2.csv')

Rec_Exp_CO2 = pd.read_csv(r'C:\Users\sxa2334\Downloads\Lithium Modeling\Code\jupyter_folder\input_csvs\New input files\Rec_export_CO2.csv')

In [ ]:
YSIOR_rec, rec_prod_cost = gp.multidict((Rec_Prod_Cost.set_index(zip(Rec_Prod_Cost['Year'], Rec_Prod_Cost['Stage'], Rec_Prod_Cost['Input'], Rec_Prod_Cost['Output'], Rec_Prod_Cost['Region'] )))['Cost'].to_dict())

YSIOR_rec, rec_fixed_cost = gp.multidict((Rec_Fixed_Cost.set_index(zip(Rec_Fixed_Cost['Year'], Rec_Fixed_Cost['Stage'], Rec_Fixed_Cost['Input'], Rec_Fixed_Cost['Output'], Rec_Fixed_Cost['Region'])))['Cost'].to_dict())

YSIOR_rec, rec_var_cost = gp.multidict((Rec_Var_Cost.set_index(zip(Rec_Var_Cost['Year'], Rec_Var_Cost['Stage'], Rec_Var_Cost['Input'], Rec_Var_Cost['Output'], Rec_Var_Cost['Region'])))['Cost'].to_dict())

YSIORR_rec, rec_import_cost = gp.multidict((Rec_Imp_Cost.set_index(zip(Pack_Imp_Cost['Year'], Pack_Imp_Cost['Stage'], Pack_Imp_Cost['Input'], Pack_Imp_Cost['Output'], Pack_Imp_Cost['RegionI'], Pack_Imp_Cost['RegionO']  )))['Cost'].to_dict())

YSIORR_rec, rec_export_cost = gp.multidict((Rec_Exp_Cost.set_index(zip(Pack_Exp_Cost['Year'], Pack_Exp_Cost['Stage'], Pack_Exp_Cost['Input'], Pack_Exp_Cost['Output'], Pack_Exp_Cost['RegionI'], Pack_Exp_Cost['RegionO']  )))['Cost'].to_dict())

In [ ]:
YSIOR_rec, rec_prod_CO2 = gp.multidict((Rec_Prod_CO2.set_index(zip(Rec_Prod_CO2['Year'], Rec_Prod_CO2['Stage'], Rec_Prod_CO2['Input'], Rec_Prod_CO2['Output'], Rec_Prod_CO2['Region'] )))['CO2'].to_dict())

YSIOR_rec, rec_fixed_CO2 = gp.multidict((Rec_Fixed_CO2.set_index(zip(Rec_Fixed_CO2['Year'], Rec_Fixed_CO2['Stage'], Rec_Fixed_CO2['Input'], Rec_Fixed_CO2['Output'], Rec_Fixed_CO2['Region'])))['CO2'].to_dict())

YSIOR_rec, rec_var_CO2 = gp.multidict((Rec_Var_CO2.set_index(zip(Rec_Var_CO2['Year'], Rec_Var_CO2['Stage'], Rec_Var_CO2['Input'], Rec_Var_CO2['Output'], Rec_Var_CO2['Region'])))['CO2'].to_dict())

YSIORR_rec, rec_import_CO2 = gp.multidict((Rec_Imp_CO2.set_index(zip(Pack_Imp_CO2['Year'], Pack_Imp_CO2['Stage'], Pack_Imp_CO2['Input'], Pack_Imp_CO2['Output'], Pack_Imp_CO2['RegionI'], Pack_Imp_CO2['RegionO']  )))['CO2'].to_dict())

YSIORR_rec, rec_export_CO2 = gp.multidict((Rec_Exp_CO2.set_index(zip(Pack_Exp_CO2['Year'], Pack_Exp_CO2['Stage'], Pack_Exp_CO2['Input'], Pack_Exp_CO2['Output'], Pack_Exp_CO2['RegionI'], Pack_Exp_CO2['RegionO']  )))['CO2'].to_dict())

## Model Set Up

In [ ]:
import os
params = {
"WLSACCESSID": os.environ.get("GUROBI_WLSACCESSID"),
"WLSSECRET": os.environ.get("GUROBI_WLSSECRET"),
"LICENSEID": int(os.environ.get("GUROBI_LICENSEID")),
}

env = gp.Env(params=params)

In [ ]:
# Declare and initialize model
m=gp.Model("Lithium", env=env)

## Decision Variables

### Production Variables

In [ ]:
#Create decision variables for the two-players model

prod_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_mine')
prod_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_proc')

prod_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_cath')
prod_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_cell')

prod_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_pack')
prod_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'prod_rec')

### Import and Export Decision Variables

In [ ]:
# change index to right ones and the name to the right ones as well (Done)
import_mine = m.addVars(YSTRR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_mine')
import_proc = m.addVars(YSIORR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_proc')
import_cath = m.addVars(YSIORR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_cath')
import_cell = m.addVars(YSIORR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_cell')
import_pack = m.addVars(YSIORR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_pack')
import_rec = m.addVars(YSIORR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_rec')

In [ ]:
# change index to right ones and the name to the right ones as well (Done)

export_mine = m.addVars(YSTRR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_mine')
export_proc = m.addVars(YSIORR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_proc')
export_cath = m.addVars(YSIORR_cath ,lb=0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_cath')
export_cell = m.addVars(YSIORR_cell ,lb= 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_cell')

export_pack = m.addVars(YSIORR_pack,lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_pack')

export_rec = m.addVars(YSIORR_rec,lb= 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_rec')

### Stockpile Decision Variables

In [ ]:
# change index to right ones and the name to the right ones as well (Done)

stockpile_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_mine')
stockpile_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_proc')
stockpile_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_cath')
stockpile_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_cell')
stockpile_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_pack')
stockpile_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'stock_rec')

### Demand Decision Variables

In [ ]:
# change index to right ones and the name to the right ones as well (Done)

demand_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'demand_mine')
demand_proc = m.addVars(YSTR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'demand_proc')
demand_cath = m.addVars(YSTR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'demand_cath')
demand_cell = m.addVars(YSTR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'demand_proc')

### Investment and Capacity Variables

In [ ]:
#Mine Investment and Capacity Constraints
active_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_mine')
invest_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_mine')
new_capacity_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_mine')
capacity_mine = m.addVars(YSTR_mine, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_mine')

In [ ]:
# Proc Investment and Capacity Constraints
active_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_proc')
invest_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_proc')
new_capacity_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_proc')
capacity_proc = m.addVars(YSIOR_proc, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_proc')

In [ ]:
# cath Investment and Capacity Constraints
active_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_cath')
invest_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_cath')
new_capacity_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_cath')
capacity_cath = m.addVars(YSIOR_cath, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_cath')

In [ ]:
# cell Investment and Capacity Constraints
active_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_cell')
invest_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_cell')
new_capacity_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_cell')
capacity_cell = m.addVars(YSIOR_cell, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_cell')

In [ ]:
# pack Investment and Capacity Constraints
active_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_pack')
invest_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_pack')
new_capacity_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_pack')
capacity_pack = m.addVars(YSIOR_pack, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_pack')

In [ ]:
# rec Investment and Capacity Constraints
active_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'active_rec')
invest_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.INTEGER, name = 'invest_rec')
new_capacity_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'new_capacity_rec')
capacity_rec = m.addVars(YSIOR_rec, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'capacity_rec')

### Deviation Variables as Needed

### Demand Constraints

In [ ]:
T_mine = ['spod', 'brine', 'clay']
T_proc = ['lce', 'loh']
T_cath = ['nmc', 'lfp']
T_cell = ['GWh_nmc', 'GWh_lfp']
T_pack = ['bev_nmc', 'bev_lfp', 'phev_nmc', 'phev_lfp']
T_rec = ['lce', 'loh']

In [ ]:
lce_li = 5.324 # 5.234 kg (or kt) LCE / kg (or kt) Li
loh_li = 3.45101
GWh_nmc_kt_li = 8.11
GWh_lfp_kt_li = 10.58
GWh_bev = 62
GWh_phev = 15
max_new_units = 25

### Demand Balance Constraints
However, the only demand for pack and recycling are parameters
All other demands are now variables
**Removed all of demand constraints except the ones for EV-pack

In [ ]:
# balance_mine_constr = m.addConstrs((
#     prod_mine.sum(y, 'mine', '*', r)
#     + gp.quicksum(import_mine[y, 'mine', t, r, r_out] for t in T_mine for r_out in R if r_out != r)
#     + gp.quicksum(stockpile_mine[(y - 1), 'mine', t, r] if (y - 1) in Y else 0 for t in T_mine)
#     >= gamma[r] * mine_demand[y, 'mine', r]
#     + gp.quicksum(export_mine[y, 'mine', t, r, r_out] for t in T_mine for r_out in R if r_out != r)
#     for y in Y for r in R
# ), name='balance_mine_constr')


In [ ]:
# balance_proc_constr= m.addConstrs((prod_proc.sum(y,'proc','*',t, r)+prod_rec.sum(y,'rec', '*',t,r)
#        + gp.quicksum(import_proc.sum(y,'proc', '*',t, r, r_out) for r_out in R if r_out != r)+ (stockpile_proc.sum((y-1),'proc','*',t,r) if y-1 in Y else 0)
#                                 >= gamma[r]*proc_demand[y,'proc',t,r] + gp.quicksum(export_proc[y,'proc',i,t, r, r_out] for i in T_mine for r_out in R if r_out != r) + stockpile_proc.sum(y, 'proc','*', t, r) 
#                                 for y in Y for t in T_proc for r in R), name='balance_proc_constr')

In [ ]:
# balance_cath_constr= m.addConstrs((prod_cath.sum(y,'cath','*',t, r)
#        + gp.quicksum(import_cath.sum(y,'cath', '*',t, r, r_out) for r_out in R if r_out != r)+ (stockpile_cath.sum((y-1),'cath','*',t,r) if y-1 in Y else 0)
#                                >= gamma[r]*cath_demand[y,'cath',t,r] + gp.quicksum(export_cath[y,'cath', i,t, r, r_out] for i in T_proc for r_out in R if r_out != r) + stockpile_cath.sum(y, 'cath','*', t, r) 
#                                 for y in Y for t in T_cath for r in R), name='balance_cath_constr')
#demand_cath_constr= m.addConstrs((prod_cath.sum(y,'cath','*',t,r)  >= gamma[r]*cath_demand[y,'cath',t,r]  for y in Y for t in T_cath for r in R) ,  name='demand_cath_constr')

In [ ]:
# balance_cell_constr= m.addConstrs((prod_cell.sum(y,'cell','*',t, r)
#        + gp.quicksum(import_cell.sum(y,'cell', '*',t, r, r_out) for r_out in R if r_out != r)+ (stockpile_cell.sum((y-1),'cell','*',t,r) if y-1 in Y else 0)
#                                >= gamma[r]*cell_demand[y,'cell',t,r] + gp.quicksum(export_cell[y,'cell', i,t, r, r_out] for i in T_cath for r_out in R if r_out != r) + stockpile_cell.sum(y, 'cell','*', t, r) 
#                                 for y in Y for t in T_cell for r in R), name='balance_cell_constr')

In [ ]:
# balance_pack_constr= m.addConstrs((prod_pack.sum(y,'pack','*',t, r)
#        + gp.quicksum(import_pack.sum(y,'pack', '*',t, r, r_out) for r_out in R if r_out!= r)+ (stockpile_pack.sum((y-1),'pack','*',t,r) if y-1 in Y else 0)
#                                 >= gamma[r]*pack_demand[y,'pack',t,r] + gp.quicksum(export_pack.sum(y,'pack','*',t, r, r_out) for r_out in R if r_out != r) + stockpile_pack.sum(y, 'pack','*', t, r) 
#                                 for y in Y for t in T_pack for r in R), name='balance_pack_constr')

In [ ]:
# balance_rec_constr= m.addConstrs((prod_rec.sum(y,'rec','*',t, r)
#        + gp.quicksum(import_rec.sum(y,'rec', '*',t, r,r_out) for r_out in R if r_out != r)+ (stockpile_rec.sum((y-1),'rec','*',t,r) if y-1 in Y else 0)
#                                 <= gamma[r]*rec_demand[y,'rec',t,r] + gp.quicksum(export_rec.sum(y,'rec','*',t, r, r_out) for r_out in R if r_out != r) + stockpile_rec.sum(y, 'rec','*', t, r) 
#                                 for y in Y for t in T_rec for r in R), name='link_rec_constr')

### Demand

In [ ]:
demand_mine_constr = m.addConstrs((prod_mine[y,'mine',t,r]
                                   + gp.quicksum(import_mine[y,'mine',t, r, r_out] for r_out in R if r_out != r)
                                   == demand_mine[y,'mine', t,r]+gp.quicksum(export_mine[y,'mine',t, r, r_out] for r_out in R if r_out != r)  for t in T_mine for y in Y for r in R) ,  name='demand_mine_constr')

prod_imp_constr_China = m.addConstrs((prod_mine[y,'mine',t,'China']
                                   >= import_mine[y,'mine',t,'US','China'] for y in Y for t in T_mine ),  name='prod_imp_constr_china')

prod_imp_constr_US = m.addConstrs((prod_mine[y,'mine',t,'US']
                                   >= import_mine[y,'mine',t,'China','US'] for y in Y for t in T_mine ),  name='prod_imp_constr_us')


In [ ]:
market_share_const= m.addConstrs((gamma[r]*prod_mine.sum(y,'mine',t,'*') == prod_mine.sum(y,'mine',t,r)
                                 for t in T_mine for y in Y_warm for r in R), name='marketshare_mine')

In [ ]:
demand_proc_constr = m.addConstrs((prod_proc.sum(y,'proc','*',t,r)
                                   + gp.quicksum(import_proc.sum(y,'proc', '*',t, r, r_out) for r_out in R if r_out != r) 
                                   == demand_proc[y,'proc',t,r]+gp.quicksum(export_proc[y,'proc',i,t, r, r_out] for i in T_mine for r_out in R if r_out != r)  for y in Y for t in T_proc for r in R) ,  name='demand_proc_constr')

proc_imp_constr_China = m.addConstrs((prod_proc.sum(y,'proc','*',t,'China') + prod_rec.sum(y,'rec', '*',t,'China') >= 
                                      import_proc.sum(y,'proc', '*',t,'US','China')
                                      for y in Y for t in T_proc) ,  name='demand_proc_constr_china')


proc_imp_constr_China = m.addConstrs((prod_proc.sum(y,'proc','*',t,'US') + prod_rec.sum(y,'rec', '*',t,'US') >= 
                                      import_proc.sum(y,'proc', '*',t,'China','US')
                                      for y in Y for t in T_proc) ,  name='demand_proc_constr_us')

In [ ]:
market_share_const= m.addConstrs((gamma[r]*prod_proc.sum(y,'proc','*',t,'*') == prod_proc.sum(y,'proc','*',t,r)
                                 for t in T_proc for y in Y_warm for r in R), name='marketshare_proc')

In [ ]:
demand_cath_constr = m.addConstrs((prod_cath.sum(y,'cath','*',t,r) 
                                   + gp.quicksum(import_cath.sum(y,'cath', '*',t, r, r_out) for r_out in R if r_out != r) 
                                   == demand_cath[y,'cath',t,r]
                                   + gp.quicksum(export_cath[y,'cath', i,t, r, r_out] for i in T_proc for r_out in R if r_out != r)  for y in Y for t in T_cath for r in R) ,  name='demand_cath_constr')

cath_imp_constr_China = m.addConstrs((prod_cath.sum(y,'cath','*',t,'China') >=
                                import_cath.sum(y,'cath', '*',t,'US','China')
                                 for y in Y for t in T_cath) ,  name='imp_cath_constr_China')
cath_imp_constr_US = m.addConstrs((prod_cath.sum(y,'cath','*',t,'US') >=
                                import_cath.sum(y,'cath', '*',t,'China','US')
                                 for y in Y for t in T_cath) ,  name='imp_cath_constr_US')

In [ ]:
market_share_const= m.addConstrs((gamma[r]*prod_cath.sum(y,'cath','*',t,'*')  == prod_cath.sum(y,'cath','*',t,r) 
                                 for t in T_cath for y in Y_warm for r in R), name='marketshare_cath')

In [ ]:
demand_cell_constr = m.addConstrs((prod_cell.sum(y,'cell','*',t,r) 
                                   + gp.quicksum(import_cell.sum(y,'cell', '*',t, r, r_out) for r_out in R if r_out != r)== 
                                   demand_cell[y,'cell',t,r]
                                   + gp.quicksum(export_cell[y,'cell', i,t, r, r_out] for i in T_cath for r_out in R if r_out != r)
                                   for y in Y for t in T_cell for r in R) ,  name='demand_cell_constr')

cell_imp_constr_china = m.addConstrs((prod_cell.sum(y,'cell','*',t,'China') >=
                                   import_cell.sum(y,'cell', '*',t,'US','China') for y in Y for t in T_cell), name='cell_imp_constr_china')

cell_imp_constr_us = m.addConstrs((prod_cell.sum(y,'cell','*',t,'US') >=
                                   import_cell.sum(y,'cell', '*',t,'China','US') for y in Y for t in T_cell), name='cell_imp_constr_us')

In [ ]:
market_share_const_cell= m.addConstrs((gamma[r]*prod_cell.sum(y,'cell','*',t,'*')  == prod_cell.sum(y,'cell','*',t,r) 
                                 for t in T_cell for y in Y_warm for r in R), name='marketshare_cell')

In [ ]:
# demand_pack_constr = m.addConstrs((prod_pack.sum(y,'pack','*',t,r)  
#                                   + gp.quicksum(import_pack.sum(y,'pack', '*',t, r, r_out) for r_out in R if r_out!= r) 
#                                    >= gamma[r]*pack_demand[y,'pack',t,r]
#                                    + gp.quicksum(export_pack.sum(y,'pack','*',t, r, r_out) 
#                                                  for r_out in R if r_out != r)  for y in Y for t in T_pack for r in R) ,  name='demand_pack_constr')
demand_pack_constr2_US = m.addConstrs((prod_pack.sum(y,'pack','*',t,'US')  
                                  + import_pack.sum(y,'pack', '*',t, 'US','China')
                                   >= 0.3*pack_demand_total[y,'pack',t]
                                   + export_pack.sum(y,'pack','*',t,'US','China') 
                                                 for y in Y for t in T_pack) ,  name='demand_pack_constr_US')

demand_pack_constr2_China = m.addConstrs((prod_pack.sum(y,'pack','*',t,'China')  
                                  + import_pack.sum(y,'pack', '*',t, 'China','US')
                                   >= 0.7*pack_demand_total[y,'pack',t]
                                   + export_pack.sum(y,'pack','*',t,'China','US') 
                                                 for y in Y for t in T_pack) ,  name='demand_pack_constr_China')


pack_imp_constr_China= m.addConstrs((prod_pack.sum(y,'pack','*',t,'China') >=
                                import_pack.sum(y,'pack', '*',t,'US','China') for y in Y for t in T_pack) ,  name='imp_pack_constr_China')

pack_imp_constr_US= m.addConstrs((prod_pack.sum(y,'pack','*',t,'US') >=
                                import_pack.sum(y,'pack', '*',t,'China','US') for y in Y for t in T_pack) ,  name='imp_pack_constr_China')

In [ ]:
market_share_const_pack= m.addConstrs((gamma[r]*prod_pack.sum(y,'pack','*',t,'*')  == prod_pack.sum(y,'pack','*',t,r) 
                                 for t in T_pack for y in Y_warm for r in R), name='marketshare_pack')

In [ ]:
# demand_rec_constr = m.addConstrs((prod_rec.sum(y,'rec','*',t,r)  
#                                   <= gamma[r]*rec_demand[y,'rec',t,r]
#                                   + gp.quicksum(export_rec.sum(y,'rec','*',t, r, r_out) for r_out in R if r_out != r)
#                                   for y in Y for t in T_rec for r in R) ,  name='demand_rec_constr')

### Production Linking Constraints

In [ ]:
link_mine_proc__lce = m.addConstrs((prod_proc[y,'proc',t,'lce',r] <= lce_li * prod_mine[y,'mine',t,r]  for y in Y for t in T_mine for r in R) ,  name='link_mine_proc_lce')

In [ ]:
link_mine_proc__loh= m.addConstrs((prod_proc[y,'proc',t,'loh',r] <= loh_li * prod_mine[y,'mine',t,r]  for y in Y for t in T_mine for r in R) ,  name='demand_link_mine_proc_loh')
# +prod_rec.sum(y,'rec','*','lce',r)

In [ ]:
link_proc_cath_nmc= m.addConstrs((loh_li * prod_cath[y,'cath',t,'nmc',r] <= prod_proc.sum(y,'proc','*','loh',r) for y in Y for t in T_proc for r in R) ,  name='demand_link_proc_cath_nmc')

In [ ]:
link_proc_cath_lfp = m.addConstrs(( lce_li * prod_cath[y,'cath',t,'lfp',r] <= prod_proc.sum(y,'proc','*','lce',r) for y in Y for t in T_proc for r in R) ,  name='demand_link_proc_cath_lfp')

In [ ]:
link_cath_cell_nmc= m.addConstrs((prod_cell[y,'cell',t,'GWh_nmc',r] <=  GWh_nmc_kt_li * prod_cath.sum(y,'cath','*','nmc',r) for y in Y for t in T_cath for r in R) ,  name='link_cath_cell_nmc')

In [ ]:
link_cath_cell_lfp= m.addConstrs((prod_cell[y,'cell',t,'GWh_lfp',r] <=  GWh_lfp_kt_li * prod_cath.sum(y,'cath','*','lfp',r) for y in Y for t in T_cath for r in R for t in T_cath) ,  name='link_cath_cell_lfp')

In [ ]:
link_cell_pack_nmc= m.addConstrs((GWh_bev * prod_pack[y,'pack',t,'bev_nmc',r] + GWh_phev * prod_pack[y,'pack',t,'phev_nmc',r] <=  prod_cell.sum(y,'cell','*','GWh_nmc',r) for y in Y for r in R for t in T_cell) ,  name='link_cell_pack_nmc')

In [ ]:
link_cell_pack_lfp = m.addConstrs((GWh_bev * prod_pack[y,'pack',t,'bev_lfp',r] + GWh_phev * prod_pack[y,'pack',t,'phev_lfp',r]<=  prod_cell.sum(y,'cell','*','GWh_lfp',r) for y in Y for t in T_cell for r in R for t in T_cell) ,  name='link_cell_pack')

In [ ]:
# link_pack_rec_nmc = m.addConstrs((prod_rec[y,'rec',t,'loh',r] <=  GWh_bev / (GWh_nmc_kt_li/loh_li) * prod_pack.sum(y,'pack','*','bev_nmc',r) + GWh_phev / (GWh_nmc_kt_li/loh_li) * prod_pack.sum(y,'pack','*','phev_nmc',r) for y in Y for t in T_pack for r in R) ,  name='link_pack_rec_nmc')

In [ ]:
# link_pack_rec_lfp = m.addConstrs((prod_rec[y,'rec',t,'lce',r] <=  GWh_bev / (GWh_lfp_kt_li/lce_li) * prod_pack.sum(y,'pack','*','bev_lfp',r) + GWh_phev / (GWh_lfp_kt_li/lce_li) * prod_pack.sum(y,'pack','*','phev_lfp',r) for y in Y for t in T_pack for r in R) ,  name='link_pack_rec_lfp')

### Import_Export Linking Constraints

In [ ]:
imp_exp_mine_US=m.addConstrs((import_mine[y,'mine', t, 'US', 'China']
                      == export_mine[y,'mine', t, 'China', 'US'] 
                                      for y in Y for t in T_mine), name='imp_exp_mine_US')
imp_exp_mine_China=m.addConstrs((import_mine[y,'mine', t, 'China', 'US']
                      == export_mine[y,'mine', t, 'US', 'China'] 
                                      for y in Y for t in T_mine), name='imp_exp_mine_China')

In [ ]:
imp_exp_proc_US=m.addConstrs((import_proc[y,'proc', t_in,t_out, 'US', 'China'] 
                           ==export_proc[y,'proc',t_in,t_out,'China', 'US'] 
                           for y in Y for t_in in T_mine for t_out in T_proc), name='imp_exp_proc_US')
imp_exp_proc_China=m.addConstrs((import_proc[y,'proc',t_in,t_out, 'China', 'US'] 
                           ==export_proc[y,'proc',t_in,t_out,'US', 'China'] 
                           for y in Y for t_in in T_mine for t_out in T_proc), name='imp_exp_proc_China')

In [ ]:
imp_exp_cath_US=m.addConstrs((import_cath[y,'cath',t_in,t_out, 'US', 'China']
                           ==export_cath[y,'cath',t_in,t_out,'China', 'US']
                             for y in Y for t_in in T_proc for t_out in T_cath), name='imp_exp_cath_US')
imp_exp_cath_China=m.addConstrs((import_cath[y,'cath',t_in,t_out,'China', 'US']
                           ==export_cath[y,'cath',t_in,t_out,'US', 'China']
                             for y in Y for t_in in T_proc for t_out in T_cath), name='imp_exp_cath_China')

In [ ]:
imp_exp_cell_US=m.addConstrs((import_cell[y,'cell',t_in,t_out, 'US', 'China']
                           ==export_cell[y,'cell',t_in,t_out,'China', 'US'] 
                       for y in Y for t_in in T_cath for t_out in T_cell), name='imp_exp_cell_US')

imp_exp_cell_China=m.addConstrs((import_cell[y,'cell',t_in,t_out, 'China', 'US']
                           ==export_cell[y,'cell',t_in,t_out,'US', 'China'] 
                       for y in Y for t_in in T_cath for t_out in T_cell), name='imp_exp_cell_China')

In [ ]:
imp_exp_pack_US=m.addConstrs((import_pack[y,'pack',t_in,t_out,'US', 'China']
                           ==export_pack[y,'pack',t_in,t_out, 'China', 'US'] 
                       for y in Y for t_in in T_cell for t_out in T_pack), name='imp_exp_pack_US')

imp_exp_pack_China=m.addConstrs((import_pack[y,'pack',t_in,t_out,'China', 'US']
                           ==export_pack[y,'pack',t_in,t_out,'US', 'China'] 
                       for y in Y for t_in in T_cell for t_out in T_pack), name='imp_exp_pack_China')

In [ ]:
# imp_exp_rec_US=m.addConstrs((import_rec.sum(y,'rec', '*',t,"US", 'China')
#                           ==export_rec.sum(y,'rec','*',t,'China','US') 
#                  for y in Y for t in T_rec), name='imp_exp_rec_US')
# imp_exp_rec_China=m.addConstrs((import_rec.sum(y,'rec', '*',t,'China', 'US')
#                           ==export_rec.sum(y,'rec','*',t,'US','China') 
#                  for y in Y for t in T_rec), name='imp_exp_rec_China')

In [ ]:
# imp_exp_mine=m.addConstrs((gp.quicksum(import_mine[y,'mine', t, r, r_out] for r_out in R if r_out != r) 
#                            == gp.quicksum(export_mine[y,'mine', t, r_out, r] for r_out in R if r_out != r) 
#                            for y in Y for t in T_mine for r in R), name='imp_exp_mine')

In [ ]:
# imp_exp_proc=m.addConstrs((gp.quicksum(import_proc.sum(y,'proc', '*',t, r_in, r) for r_in in R if r_in != r)
#                            ==gp.quicksum(export_proc[y,'proc',i,t, r, r_out] for i in T_mine for r_out in R if r_out != r) 
#                            for y in Y for t in T_proc for r in R), name='imp_exp_proc')

In [ ]:
# imp_exp_cath=m.addConstrs((gp.quicksum(import_cath.sum(y,'cath', '*',t, r_in, r) for r_in in R if r_in != r)
#                            ==gp.quicksum(export_cath[y,'cath', i,t, r, r_out] 
#                             for i in T_proc for r_out in R if r_out != r) for y in Y for t in T_cath for r in R), name='imp_exp_cath')

In [ ]:
# imp_exp_cell=m.addConstrs((gp.quicksum(import_cell.sum(y,'cell', '*',t, r_in, r) for r_in in R if r_in != r)
#                            ==gp.quicksum(export_cell[y,'cell', i,t, r, r_out] 
#                         for i in T_cath for r_out in R if r_out != r) for y in Y for t in T_cell for r in R), name='imp_exp_cell')

In [ ]:
# imp_exp_pack=m.addConstrs((gp.quicksum(import_pack.sum(y,'pack', '*',t, r_in, r) for r_in in R if r_in != r)
#                            ==gp.quicksum(export_pack.sum(y,'pack','*',t, r, r_out) 
#                         for r_out in R if r_out != r) for y in Y for t in T_pack for r in R), name='imp_exp_pack')

In [ ]:
# imp_exp_rec=m.addConstrs((gp.quicksum(import_rec.sum(y,'rec', '*',t, r_in, r) for r_in in R if r_in != r)
#                           ==gp.quicksum(export_rec.sum(y,'rec','*',t, r, r_out) 
#                                 for r_out in R if r_out != r) for y in Y for t in T_rec for r in R), name='imp_exp_rec')

### Zero Constraints

In [ ]:
# zero_mine_us=m.addConstrs((prod_mine.sum(y,'mine',t,'US')==0 for y in Y for t in T_mine), name='zero_mine_us')
# zer_proc_us=m.addConstrs((prod_proc.sum(y,'proc','*',t,'US')==0 for y in Y for t in T_proc), name='zero_proc_us')    
# zer_cath_us=m.addConstrs((prod_cath.sum(y,'cath','*',t,'US')==0 for y in Y for t in T_cath), name='zero_cath_us')
# zer_cell_us=m.addConstrs((prod_cell.sum(y,'cell','*',t,'US')==0 for y in Y for t in T_cell), name='zero_cell_us')
# zer_pack_us=m.addConstrs((prod_pack.sum(y,'pack','*',t,'US')==0 for y in Y for t in T_pack), name='zero_pack_us')
# zer_rec_us=m.addConstrs((prod_rec.sum(y,'rec','*',t, 'US')==0 for y in Y for t in T_rec), name='zero_rec_us')

In [ ]:
# zero_stockpile_mine=m.addConstrs((stockpile_mine.sum(y,'mine',t,r)==0 for y in Y for t in T_mine for r in R), name='zero_stock_mine')
# zero_stockpile_proc=m.addConstrs((stockpile_proc.sum(y,'proc',t,r)==0 for y in Y for t in T_mine for r in R), name='zero_stock_proc')
# zero_stockpile_cath=m.addConstrs((stockpile_cath.sum(y,'cath',t,r)==0 for y in Y for t in T_mine for r in R), name='zero_stock_cath')
# zero_stockpile_pack=m.addConstrs((stockpile_pack.sum(y,'pack',t,r)==0 for y in Y for t in T_mine for r in R), name='zero_stock_pack')
# zero_stockpile_rec=m.addConstrs((stockpile_rec.sum(y,'rec',t,r)==0 for y in Y for t in T_mine for r in R), name='zero_stock_rec')

### Mine Investment and Capacity Constraints

In [ ]:
# 25 kt Li / year max
max_cap_mine = 25

In [ ]:
#Spod Initial Mines
initial_mine_spod_constr = m.addConstrs((active_mine[y,'mine', 'spod','China'] == 4 for y in Y_start for r in R), name = 'initial_mine_spod_constr')
initial_mine_capacity_spod_constr =m.addConstrs((capacity_mine[y,'mine', 'spod','China'] == 100 for y in Y_start), name = 'initial_mine_capacity_spod_constr')

#Spod Initial Mines
initial_mine_spod_constr = m.addConstrs((active_mine[y,'mine', 'spod','US'] == 0 for y in Y_start for r in R), name = 'initial_mine_spod_constr')
initial_mine_capacity_spod_constr =m.addConstrs((capacity_mine[y,'mine', 'spod','US'] == 0 for y in Y_start), name = 'initial_mine_capacity_spod_constr')

# Brine Intial Mines
initial_mine_brine_constr = m.addConstrs((active_mine[y,'mine', 'brine','US'] == 4 for y in Y_start), name = 'initial_mine_brine_constr')
initial_mine_capacity_brine_constr = m.addConstrs((capacity_mine[y,'mine', 'brine',"US"] == 100 for y in Y_start), name = 'initial_mine_capacity_brine_constr')

# Brine Intial Mines
initial_mine_brine_constr = m.addConstrs((active_mine[y,'mine', 'brine','China'] == 0 for y in Y_start), name = 'initial_mine_brine_constr')
initial_mine_capacity_brine_constr = m.addConstrs((capacity_mine[y,'mine', 'brine',"China"] == 0 for y in Y_start), name = 'initial_mine_capacity_brine_constr')


# Clay Initial Mines
initial_mine_clay_constr = m.addConstrs((active_mine[y,'mine', 'clay',r] == 0 for y in Y_start for r in R), name = 'initial_mine_clay_constr')
inital_mine_capacity_clay_constr = m.addConstrs((capacity_mine[y,'mine', 'clay',r] == 0 for y in Y_start for r in R), name = 'initial_mine_capacity_clay_constr')

In [ ]:
active_mine_constr = m.addConstrs((active_mine[y,'mine', t, r] == active_mine[(y-1),'mine',t,r] + invest_mine[(y-5),'mine',t,r]  for y in Y_warm for t in T_mine for r in R) ,  name='active_mine_constr')

In [ ]:
invest_mine_constr = m.addConstrs((new_capacity_mine[y,'mine', t, r] <= beta[r]*max_cap_mine * invest_mine[(y),'mine',t, r] for y in Y for t in T_mine for r in R) ,  name='invest_mine_constr')

In [ ]:
new_capacity_mine_constr = m.addConstrs((capacity_mine[y,'mine', t, r] <= new_capacity_mine[(y-5),'mine',t, r] + capacity_mine[(y-1),'mine',t, r] for y in Y_warm for t in T_mine for r in R) ,  name='new_capacity_mine_constr')

In [ ]:
capacity_mine_constr = m.addConstrs((prod_mine[y,'mine', t, r] <= capacity_mine[(y),'mine',t, r] for y in Y for t in T_mine for r in R) ,  name='capacity_mine_constr')

In [ ]:
max_new_units_mine_constr = m.addConstrs(invest_mine.sum(y,'mine','*', r) <= beta[r]*max_new_units for y in Y for r in R)

In [ ]:
# kt Li by weight
resources_brine = 40000
resources_spod = 40000
resources_clay = 20000

In [ ]:
resources_brine_constr = m.addConstrs((prod_mine.sum('*','mine', 'brine', r) <= beta[r]*resources_brine for r in R) ,  name='resources_brine_constr')
resources_spod_constr = m.addConstrs((prod_mine.sum('*','mine', 'spod', r) <= beta[r]*resources_spod for r in R) ,  name='resources_spod_constr')
resources_clay_constr = m.addConstrs((prod_mine.sum('*','mine', 'clay', r) <=beta[r]*resources_brine for r in R) ,  name='resources_clay_constr')

### Processing Capacity and Investment

In [ ]:
# kt of loh or lce
max_cap_proc = 100

In [ ]:
#Spod to LOH Initial procs
initial_proc_spod_loh_constr = m.addConstrs((active_proc[y,'proc', 'spod', 'loh', 'China'] == 4 for y in Y_start), name = 'initial_proc_spod_loh_constr')
initial_proc_capacity_spod_loh_constr =m.addConstrs((capacity_proc[y,'proc', 'spod', 'loh', 'China'] == 400 for y in Y_start), name = 'initial_proc_capacity_spod_loh_constr')

#Spod to LOH Initial procs
# initial_proc_spod_loh_constr = m.addConstrs((active_proc[y,'proc', 'spod', 'loh', 'US'] == 0 for y in Y_start), name = 'initial_proc_spod_loh_constr')
# initial_proc_capacity_spod_loh_constr =m.addConstrs((capacity_proc[y,'proc', 'spod', 'loh', 'US'] == 0 for y in Y_start), name = 'initial_proc_capacity_spod_loh_constr')

# Brine to LCE Intial procs
initial_proc_brine_lce_constr = m.addConstrs((active_proc[y,'proc', 'brine', 'lce', 'China'] == 4 for y in Y_start), name = 'initial_proc_brine_lce_constr')
initial_proc_capacity_brine_lce_constr =m.addConstrs((capacity_proc[y,'proc', 'brine', 'lce', 'China'] == 400 for y in Y_start), name = 'initial_proc_capacity_brine_lce_constr')

# initial_proc_brine_lce_constr = m.addConstrs((active_proc[y,'proc', 'brine', 'lce', 'US'] == 0 for y in Y_start), name = 'initial_proc_brine_lce_constr')
# initial_proc_capacity_brine_lce_constr =m.addConstrs((capacity_proc[y,'proc', 'brine', 'lce', 'US'] == 0 for y in Y_start), name = 'initial_proc_capacity_brine_lce_constr')

# Setting Zeros
initial_proc_zero_constr = m.addConstrs((active_proc[y,'proc', 'clay', 'lce', r] + active_proc[y,'proc', 'clay', 'loh',r] + active_proc[y,'proc', 'spod', 'lce',r] + active_proc[y,'proc', 'brine', 'loh', r] == 0 for y in Y_start for r in R), name = 'initial_proc_zero_constr')
inital_proc_capacity_zero_constr = m.addConstrs((capacity_proc[y,'proc', 'clay', 'lce', r] + capacity_proc[y,'proc', 'clay', 'loh',r] + capacity_proc[y,'proc', 'spod', 'lce', r] + capacity_proc[y,'proc', 'brine', 'loh',r] == 0 for y in Y_start for r in R), name = 'initial_proc_capacity_zero_constr')

In [ ]:
active_proc_constr = m.addConstrs((active_proc[y,'proc',i, t, r] == active_proc[(y-1),'proc', i, t, r] + invest_proc[(y-5),'proc',i, t, r]  for y in Y_warm for i in T_mine for t in T_proc for r in R) ,  name='active_proc_constr')

In [ ]:
invest_proc_constr = m.addConstrs((new_capacity_proc[y,'proc',i, t, r] <=max_cap_proc * invest_proc[(y),'proc',i,t, r] for y in Y for i in T_mine for t in T_proc for r in R) ,  name='invest_proc_constr')

In [ ]:
new_capacity_proc_constr = m.addConstrs((capacity_proc[y,'proc',i, t, r] <= new_capacity_proc[(y-5),'proc',i,t, r] + capacity_proc[(y-1),'proc',i,t, r] for y in Y_warm for i in T_mine for t in T_proc for r in R) ,  name='new_capacity_proc_constr')

In [ ]:
capacity_proc_constr = m.addConstrs((prod_proc[y,'proc', i,t,r]<= capacity_proc[(y),'proc',i,t,r] for y in Y for i in T_mine for t in T_proc for r in R) ,  name='capacity_proc_constr')

In [ ]:
max_new_units_proc_constr = m.addConstrs(invest_proc.sum(y,'proc','*', r) <=beta[r]*max_new_units for y in Y for r in R)

### Cathode Capacity and Investment

In [ ]:
# kt of cathode active materials
max_cap_cath = 12

In [ ]:
#loh to nmc Initial caths
initial_cath_loh_nmc_constr = m.addConstrs((active_cath[y,'cath', 'loh', 'nmc', 'China'] == 10 for y in Y_start), name = 'initial_cath_loh_nmc_constr')
initial_cath_capacity_loh_nmc_constr =m.addConstrs((capacity_cath[y,'cath', 'loh', 'nmc', 'China'] == 100 for y in Y_start), name = 'initial_cath_capacity_loh_nmc_constr')

# initial_cath_loh_nmc_constr = m.addConstrs((active_cath[y,'cath', 'loh', 'nmc', 'US'] == 0 for y in Y_start), name = 'initial_cath_loh_nmc_constr')
# initial_cath_capacity_loh_nmc_constr =m.addConstrs((capacity_cath[y,'cath', 'loh', 'nmc', 'US'] == 0 for y in Y_start), name = 'initial_cath_capacity_loh_nmc_constr')
    
# lce to lfp Intial caths
initial_cath_lce_lfp_constr = m.addConstrs((active_cath[y,'cath', 'lce', 'lfp', 'China'] ==5 for y in Y_start), name = 'initial_cath_lce_lfp_constr')
initial_cath_capacity_lce_lfp_constr =m.addConstrs((capacity_cath[y,'cath', 'lce', 'lfp','China'] == 50 for y in Y_start), name = 'initial_cath_capacity_lce_lfp_constr')

# initial_cath_lce_lfp_constr = m.addConstrs((active_cath[y,'cath', 'lce', 'lfp', 'US'] == 0 for y in Y_start), name = 'initial_cath_lce_lfp_constr')
# initial_cath_capacity_lce_lfp_constr =m.addConstrs((capacity_cath[y,'cath', 'lce', 'lfp','US'] == 0 for y in Y_start), name = 'initial_cath_capacity_lce_lfp_constr')


# Setting Zeros
initial_cath_zero_constr = m.addConstrs((active_cath[y,'cath', 'loh', 'lfp',r] + active_cath[y,'cath', 'lce', 'nmc',r]  == 0 for y in Y_start for r in R), name = 'initial_cath_zero_constr')
inital_cath_capacity_zero_constr = m.addConstrs((capacity_cath[y,'cath', 'loh', 'lfp',r] + capacity_cath[y,'cath', 'lce', 'nmc',r] == 0 for y in Y_start for r in R), name = 'initial_cath_capacity_zero_constr')

In [ ]:
active_cath_constr = m.addConstrs((active_cath[y,'cath',i, t,r] == active_cath[(y-1),'cath', i, t,r] + invest_cath[(y-5),'cath',i, t, r]  for y in Y_warm for i in T_proc for t in T_cath for r in R) ,  name='active_cath_constr')

In [ ]:
invest_cath_constr = m.addConstrs((new_capacity_cath[y,'cath',i, t,r] <= beta[r]*max_cap_cath * invest_cath[(y),'cath',i,t,r] for y in Y for i in T_proc for t in T_cath for r in R) ,  name='invest_cath_constr')

In [ ]:
new_capacity_cath_constr = m.addConstrs((capacity_cath[y,'cath',i, t,r] <= new_capacity_cath[(y-5),'cath',i,t,r] + capacity_cath[(y-1),'cath',i,t,r] for y in Y_warm for i in T_proc for t in T_cath for r in R) ,  name='new_capacity_cath_constr')

In [ ]:
capacity_cath_constr = m.addConstrs((prod_cath[y,'cath', i,t,r] <= capacity_cath[(y),'cath',i,t,r] for y in Y for i in T_proc for t in T_cath for r in R) ,  name='capacity_cath_constr')

In [ ]:
max_new_units_cath_constr = m.addConstrs(invest_cath.sum(y,'cath','*',r) <= beta[r]*max_new_units for y in Y for r in R)

### Cell Capacity and Investment

In [ ]:
# GWh of cells
max_cap_cell = 100

In [ ]:
#nmc to GWh_nmc Initial cells
initial_cell_nmc_GWh_nmc_constr = m.addConstrs((active_cell[y,'cell', 'nmc', 'GWh_nmc','China'] == 15 for y in Y_start), name = 'initial_cell_nmc_GWh_nmc_constr')
initial_cell_nmc_GWh_nmc_constr = m.addConstrs((active_cell[y,'cell', 'nmc', 'GWh_nmc','US'] == 5 for y in Y_start), name = 'initial_cell_nmc_GWh_nmc_constr')

initial_cell_capacity_nmc_GWh_nmc_constr =m.addConstrs((capacity_cell[y,'cell', 'nmc', 'GWh_nmc',r] == 1500 for y in Y_start for r in R), name = 'initial_cell_capacity_nmc_GWh_nmc_constr')
    
# lfp to GWh_lfp Intial cells
initial_cell_lfp_GWh_lfp_constr = m.addConstrs((active_cell[y,'cell', 'lfp', 'GWh_lfp','China'] == 5 for y in Y_start), name = 'initial_cell_lfp_GWh_lfp_constr')
initial_cell_capacity_lfp_GWh_lfp_constr =m.addConstrs((capacity_cell[y,'cell', 'lfp', 'GWh_lfp','China'] == 1000 for y in Y_start), name = 'initial_cell_capacity_lfp_GWh_lfp_constr')

# initial_cell_lfp_GWh_lfp_constr = m.addConstrs((active_cell[y,'cell', 'lfp', 'GWh_lfp','US'] == 0 for y in Y_start), name = 'initial_cell_lfp_GWh_lfp_constr')
# initial_cell_capacity_lfp_GWh_lfp_constr =m.addConstrs((capacity_cell[y,'cell', 'lfp', 'GWh_lfp','US'] == 0 for y in Y_start), name = 'initial_cell_capacity_lfp_GWh_lfp_constr')


# Setting Zeros
initial_cell_zero_constr = m.addConstrs((active_cell[y,'cell', 'nmc', 'GWh_lfp',r] + active_cell[y,'cell', 'lfp', 'GWh_nmc',r]  == 0 for y in Y_start for r in R), name = 'initial_cell_zero_constr')
inital_cell_capacity_zero_constr = m.addConstrs((capacity_cell[y,'cell', 'nmc', 'GWh_lfp', r] + capacity_cell[y,'cell', 'lfp', 'GWh_nmc', r] == 0 for y in Y_start for r in R), name = 'initial_cell_capacity_zero_constr')

In [ ]:
active_cell_constr = m.addConstrs((active_cell[y,'cell',i, t,r] == active_cell[(y-1),'cell', i, t,r] + invest_cell[(y-5),'cell',i, t, r]  for y in Y_warm for i in T_cath for t in T_cell for r in R) ,  name='active_cell_constr')

In [ ]:
invest_cell_constr = m.addConstrs((new_capacity_cell[y,'cell',i, t, r] <=beta[r]*max_cap_cell * invest_cell[(y),'cell',i,t,r] for y in Y for i in T_cath for t in T_cell for r in R) ,  name='invest_cell_constr')

In [ ]:
new_capacity_cell_constr = m.addConstrs((capacity_cell[y,'cell',i, t, r] <= new_capacity_cell[(y-5),'cell',i,t, r] + capacity_cell[(y-1),'cell',i,t, r] for y in Y_warm for i in T_cath for t in T_cell for r in R) ,  name='new_capacity_cell_constr')

In [ ]:
capacity_cell_constr = m.addConstrs((prod_cell[y,'cell', i,t, r] <= capacity_cell[y,'cell',i,t, r] for y in Y for i in T_cath for t in T_cell for r in R) ,  name='capacity_cell_constr')

In [ ]:
max_new_units_cell_constr = m.addConstrs(invest_cell.sum(y,'cell','*', r) <=beta[r]*max_new_units for y in Y for r in R)

### Pack Capacity and Investment

In [ ]:
# MM of EVs BEVs and PHEVs
max_cap_pack = 1

In [ ]:
#nmc to bev_nmc Initial packs
initial_pack_GWh_nmc_bev_nmc_constr = m.addConstrs((active_pack[y,'pack', 'GWh_nmc', 'bev_nmc','China'] ==8 for y in Y_start), name = 'initial_pack_nmc_bev_nmc_constr')

initial_pack_GWh_nmc_bev_nmc_constr = m.addConstrs((active_pack[y,'pack', 'GWh_nmc', 'bev_nmc','US'] ==2 for y in Y_start), name = 'initial_pack_nmc_bev_nmc_constr')

initial_pack_capacity_GWh_nmc_bev_nmc_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_nmc', 'bev_nmc',r] == 10 for y in Y_start for r in R), name = 'initial_pack_capacity_nmc_bev_nmc_constr')
    
# lfp to bev_lfp Intial packs
initial_pack_GWh_lfp_bev_lfp_constr = m.addConstrs((active_pack[y,'pack', 'GWh_lfp', 'bev_lfp','China'] == 5 for y in Y_start), name = 'initial_pack_lfp_bev_lfp_constr')
initial_pack_capacity_GWh_lfp_bev_lfp_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_lfp', 'bev_lfp','China'] == 5 for y in Y_start), name = 'initial_pack_capacity_lfp_bev_lfp_constr')

# initial_pack_GWh_lfp_bev_lfp_constr = m.addConstrs((active_pack[y,'pack', 'GWh_lfp', 'bev_lfp','US'] == 0 for y in Y_start), name = 'initial_pack_lfp_bev_lfp_constr')
# initial_pack_capacity_GWh_lfp_bev_lfp_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_lfp', 'bev_lfp','US'] == 0 for y in Y_start), name = 'initial_pack_capacity_lfp_bev_lfp_constr')

#nmc to phev_nmc Initial packs
initial_pack_GWh_nmc_phev_nmc_constr = m.addConstrs((active_pack[y,'pack', 'GWh_nmc', 'phev_nmc','China'] ==5 for y in Y_start), name = 'initial_pack_nmc_phev_nmc_constr')
initial_pack_capacity_GWh_nmc_phev_nmc_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_nmc', 'phev_nmc','China'] == 5 for y in Y_start), name = 'initial_pack_capacity_nmc_phev_nmc_constr')

# #nmc to phev_nmc Initial packs
# initial_pack_GWh_nmc_phev_nmc_constr = m.addConstrs((active_pack[y,'pack', 'GWh_nmc', 'phev_nmc','US'] == 0 for y in Y_start), name = 'initial_pack_nmc_phev_nmc_constr')
# initial_pack_capacity_GWh_nmc_phev_nmc_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_nmc', 'phev_nmc','US'] == 0 for y in Y_start), name = 'initial_pack_capacity_nmc_phev_nmc_constr')

# lfp to phev_lfp Intial packs
initial_pack_GWh_lfp_phev_lfp_constr = m.addConstrs((active_pack[y,'pack', 'GWh_lfp', 'phev_lfp','China'] ==2 for y in Y_start), name = 'initial_pack_lfp_phev_lfp_constr')
initial_pack_capacity_GWh_lfp_phev_lfp_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_lfp', 'phev_lfp','China'] ==2 for y in Y_start), name = 'initial_pack_capacity_lfp_phev_lfp_constr')

# # lfp to phev_lfp Intial packs
# initial_pack_GWh_lfp_phev_lfp_constr = m.addConstrs((active_pack[y,'pack', 'GWh_lfp', 'phev_lfp','US'] == 0 for y in Y_start), name = 'initial_pack_lfp_phev_lfp_constr')
# initial_pack_capacity_GWh_lfp_phev_lfp_constr =m.addConstrs((capacity_pack[y,'pack', 'GWh_lfp', 'phev_lfp','US'] == 0 for y in Y_start), name = 'initial_pack_capacity_lfp_phev_lfp_constr')

# Setting Zeros
initial_pack_zero_constr = m.addConstrs((active_pack[y,'pack', 'GWh_nmc', 'bev_lfp',r] + active_pack[y,'pack', 'GWh_lfp', 'bev_nmc',r] + active_pack[y,'pack', 'GWh_nmc', 'phev_lfp',r] + active_pack[y,'pack', 'GWh_lfp', 'phev_nmc',r]  == 0 for y in Y_start for r in R), name = 'initial_pack_zero_constr')
inital_pack_capacity_zero_constr = m.addConstrs((capacity_pack[y,'pack', 'GWh_nmc', 'bev_lfp',r] + capacity_pack[y,'pack', 'GWh_lfp', 'bev_nmc',r] + capacity_pack[y,'pack', 'GWh_nmc', 'phev_lfp',r] + capacity_pack[y,'pack', 'GWh_lfp', 'phev_nmc',r] == 0 for y in Y_start for r in R), name = 'initial_pack_capacity_zero_constr')

In [ ]:
active_pack_constr = m.addConstrs((active_pack[y,'pack',i, t, r] == active_pack[(y-1),'pack', i, t, r] + invest_pack[(y-5),'pack',i, t, r]  for y in Y_warm for i in T_cell for t in T_pack for r in R) ,  name='active_pack_constr')

In [ ]:
invest_pack_constr = m.addConstrs((new_capacity_pack[y,'pack',i, t,r] <=beta[r]*max_cap_pack * invest_pack[(y),'pack',i,t,r] for y in Y for i in T_cell for t in T_pack for r in R) ,  name='invest_pack_constr')

In [ ]:
new_capacity_pack_constr = m.addConstrs((capacity_pack[y,'pack',i, t,r] <= new_capacity_pack[(y-5),'pack',i,t,r] + capacity_pack[(y-1),'pack',i,t,r] for y in Y_warm for i in T_cell for t in T_pack for r in R) ,  name='new_capacity_pack_constr')

In [ ]:
capacity_pack_constr = m.addConstrs((prod_pack[y,'pack', i,t,r] <= capacity_pack[(y),'pack',i,t,r] for y in Y for i in T_cell for t in T_pack for r in R) ,  name='capacity_pack_constr')

In [ ]:
max_new_units_pack_constr = m.addConstrs(invest_pack.sum(y,'pack','*',r) <= beta[r]*max_new_units for y in Y for r in R)

### Recycling Capacity and Investment

In [ ]:
# kt of LCE and LOH
max_cap_rec = 15

In [ ]:
# Setting Zeros
initial_rec_zero_constr = m.addConstrs((active_rec[y,'rec', 'bev_nmc', 'lce',r] + active_rec[y,'rec', 'bev_lfp', 'loh',r] + active_rec[y,'rec', 'bev_nmc', 'loh',r] + active_rec[y,'rec', 'bev_lfp', 'lce',r] +
                                        active_rec[y,'rec', 'phev_nmc', 'lce',r] + active_rec[y,'rec', 'phev_lfp', 'loh',r] + active_rec[y,'rec', 'phev_nmc', 'loh',r] + active_rec[y,'rec', 'phev_lfp', 'lce',r]  == 0 for y in Y_start for r in R), name = 'initial_rec_zero_constr')

initial_rec_capacity_zero_constr = m.addConstrs((capacity_rec[y,'rec', 'bev_nmc', 'lce',r] + capacity_rec[y,'rec', 'bev_lfp', 'loh',r] + capacity_rec[y,'rec', 'bev_nmc', 'loh',r] + capacity_rec[y,'rec', 'bev_lfp', 'lce',r] +
                                        capacity_rec[y,'rec', 'phev_nmc', 'lce',r] + capacity_rec[y,'rec', 'phev_lfp', 'loh',r] + capacity_rec[y,'rec', 'phev_nmc', 'loh',r] + capacity_rec[y,'rec', 'phev_lfp', 'lce',r]  == 0 for y in Y_start for r in R), name = 'initial_rec_capacity_zero_constr')

In [ ]:
active_rec_constr = m.addConstrs((active_rec[y,'rec',i, t,r] == active_rec[(y-1),'rec', i, t,r] + invest_rec[(y-5),'rec',i, t,r]  for y in Y_warm for i in T_pack for t in T_rec for r in R) ,  name='active_rec_constr')

In [ ]:
invest_rec_constr = m.addConstrs((new_capacity_rec[y,'rec',i, t,r] <= beta[r]*max_cap_rec * invest_rec[(y),'rec',i,t,r] for y in Y for i in T_pack for t in T_rec for r in R) ,  name='invest_rec_constr')

In [ ]:
new_capacity_rec_constr = m.addConstrs((capacity_rec[y,'rec',i, t,r] <= new_capacity_rec[(y-5),'rec',i,t,r] + capacity_rec[(y-1),'rec',i,t,r] for y in Y_warm for i in T_pack for t in T_rec for r in R) ,  name='new_capacity_rec_constr')

In [ ]:
capacity_rec_constr = m.addConstrs((prod_rec[y,'rec', i,t,r] <= capacity_rec[(y),'rec',i,t,r] for y in Y for i in T_pack for t in T_rec for r in R) ,  name='capacity_rec_constr')

In [ ]:
max_new_units_rec_constr = m.addConstrs(invest_rec.sum(y,'rec','*',r) <= beta[r]*max_new_units for y in Y for r in R)

## Objective Function

### Cost Variables (add cost for import, export, stockpile)

In [ ]:
mine_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'mine_cost')
mine_cost_constr = m.addConstrs((
    mine_cost[r] == alpha[r] * (
        quicksum(prod_mine[y, 'mine', t, r]* mine_prod_cost[y,'mine', t, r] for y in Y for t in T_mine) +
        quicksum(new_capacity_mine[y, 'mine',t, r] * mine_var_cost[y,'mine',t, r] for y in Y for t in T_mine) +
        quicksum(invest_mine[y,'mine',t, r] * mine_fixed_cost[y,'mine',t, r] for y in Y for t in T_mine)
    ) 
    for r in R
), name='mine_cost_constr')                                                  

In [ ]:
proc_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'proc_cost')
proc_cost_constr = m.addConstrs((
    proc_cost[r] == alpha[r] * (
        quicksum(prod_proc[y,'proc', i, o, r] * proc_prod_cost[y,'proc', i, o, r] for y in Y for i in T_mine for o in T_proc) +
        quicksum(new_capacity_proc[y, 'proc', i, o, r] * proc_var_cost[y, 'proc', i, o, r] for y in Y for i in T_mine for o in T_proc) +
        quicksum(invest_proc[y,'proc', i, o, r] * proc_fixed_cost[y,'proc', i, o, r] for y in Y for i in T_mine for o in T_proc)
    )
    for r in R
), name='proc_cost_constr')
#proc_cost_constr = m.addConstrs((proc_cost[r] == alpha[r]*(prod_proc.prod(proc_prod_cost) + new_capacity_proc.prod(proc_var_cost) + invest_proc.prod(proc_fixed_cost)) for r in R),  name='proc_cost_constr')

In [ ]:
cath_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'cath_cost')

cath_cost_constr = m.addConstrs((
    cath_cost[r] == alpha[r] * (
        quicksum(prod_cath[y,'cath', i, o, r] * cath_prod_cost[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath) +
        quicksum(new_capacity_cath[y,'cath', i, o, r] * cath_var_cost[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath) +
        quicksum(invest_cath[y,'cath', i, o, r] * cath_fixed_cost[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath)
    )
    for r in R
), name='cath_cost_constr')

# cath_cost_constr = m.addConstrs((cath_cost[r] == alpha[r]*(prod_cath.prod(cath_prod_cost) + new_capacity_cath.prod(cath_var_cost) + invest_cath.prod(cath_fixed_cost)) for r in R),  name='cath_cost_constr')

In [ ]:
cell_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'cell_cost')

cell_cost_constr = m.addConstrs((
    cell_cost[r] == alpha[r] * (
        quicksum(prod_cell[y,'cell', i, o, r] * cell_prod_cost[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell) +
        quicksum(new_capacity_cell[y,'cell', i, o, r] * cell_var_cost[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell) +
        quicksum(invest_cell[y,'cell', i, o, r] * cell_fixed_cost[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell)
    )
    for r in R
), name='cell_cost_constr')
# cell_cost_constr = m.addConstrs((cell_cost[r] == alpha[r]*(prod_cell.prod(cell_prod_cost) + new_capacity_cell.prod(cell_var_cost) + invest_cell.prod(cell_fixed_cost)) for r in R),  name='cell_cost_constr')

In [ ]:
pack_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'pack_cost')

pack_cost_constr = m.addConstrs((
    pack_cost[r] == alpha[r] * (
        quicksum(prod_pack[y,'pack', i, o, r] * pack_prod_cost[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack) +
        quicksum(new_capacity_pack[y,'pack', i, o, r] * pack_var_cost[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack) +
        quicksum(invest_pack[y,'pack', i, o, r] * pack_fixed_cost[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack)
    )
    for r in R
), name='pack_cost_constr')
# pack_cost_constr = m.addConstrs((pack_cost[r] == alpha[r]*(prod_pack.prod(pack_prod_cost) + new_capacity_pack.prod(pack_var_cost) + invest_pack.prod(pack_fixed_cost)) for r in R),  name='pack_cost_constr')

In [ ]:
rec_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'rec_cost')
rec_cost_constr = m.addConstrs((
    rec_cost[r] == alpha[r] * (
        quicksum(prod_rec[y,'rec', i, o, r] * rec_prod_cost[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec) +
        quicksum(new_capacity_rec[y,'rec', i, o, r] * rec_var_cost[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec) +
        quicksum(invest_rec[y,'rec', i, o, r] * rec_fixed_cost[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec)
    )
    for r in R
), name='rec_cost_constr')
# rec_cost_constr = m.addConstrs((rec_cost[r] == alpha[r]*(prod_rec.prod(rec_prod_cost) + new_capacity_rec.prod(rec_var_cost) + invest_rec.prod(rec_fixed_cost)) for r in R),  name='rec_cost_constr')

In [ ]:
import_cost=m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_cost')
import_cost_constr=m.addConstrs((
    import_cost[r]==imp[r]*(
        quicksum(import_mine[y,'mine', t, r,r_out]*mine_imp_cost[y,'mine',t, r,r_out] for y in Y for t in T_mine for r_out in R if r_out != r)+
        quicksum(import_proc[y,'proc',i,o, r, r_out]*proc_imp_cost[y,'proc',i,o, r, r_out] for y in Y for i in T_mine for o in T_proc for r_out in R if r_out != r)+
        quicksum(import_cath[y,'cath',i,o, r,r_out]*cath_imp_cost[y,'cath',i,o, r, r_out] for y in Y for i in T_proc for o in T_cath for r_out in R if r_out != r)+
        quicksum(import_cell[y,'cell',i,o, r, r_out]*cell_import_cost[y,'cell',i,o, r,r_out] for y in Y for i in T_cath for o in T_cell for r_out in R if r_out != r)+
        quicksum(import_pack[y,'pack',i,o, r, r_out]*pack_import_cost[y,'pack',i,o, r, r_out] for y in Y for i in T_cell for o in T_pack for r_out in R if r_out != r)+
        quicksum(import_rec.sum(y,'rec','*',o, r, r_out)*rec_import_cost.sum(y,'rec','*',o, r, r_out) for y in Y for o in T_rec for r_out in R if r_out != r)
                                )
                                 for r in R), name='import_cost')
# import_cost_constr=m.addConstrs((import_cost[r]==imp[r]*(import_mine.prod(mine_imp_cost)+import_proc.prod(proc_imp_cost)+import_cath.prod(cath_imp_cost)+import_cell.prod(cell_import_cost)+import_pack.prod(pack_import_cost)+import_rec.prod(rec_import_cost)) for r in R), name='import_cost')

In [ ]:
export_cost_main=m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_cost_main')
export_cost_constr=m.addConstrs((
    export_cost_main[r]==imp[r]*(
        quicksum(export_mine[y,'mine', t, r, r_out]*mine_exp_cost[y,'mine',t, r, r_out] for y in Y for t in T_mine for r_out in R if r_out != r)+
        quicksum(export_proc[y,'proc',i,o, r, r_out]*proc_exp_cost[y,'proc',i,o, r, r_out] for y in Y for i in T_mine for o in T_proc for r_out in R if r_out != r)+
        quicksum(export_cath[y,'cath',i,o, r, r_out]*cath_exp_cost[y,'cath',i,o, r,r_out] for y in Y for i in T_proc for o in T_cath for r_out in R if r_out != r)+
        quicksum(export_cell[y,'cell',i,o, r,r_out]*cell_export_cost[y,'cell',i,o, r, r_out] for y in Y for i in T_cath for o in T_cell for r_out in R if r_out != r)+
        quicksum(export_pack[y,'pack',i,o, r, r_out]*pack_export_cost[y,'pack',i,o, r, r_out] for y in Y for i in T_cell for o in T_pack for r_out in R if r_out != r)+
        quicksum(export_rec.sum(y,'rec','*',o, r,r_out)*rec_export_cost.sum(y,'rec','*',o, r,r_out) for y in Y for o in T_rec for r_out in R if r_out != r)
                                )
                                 for r in R), name='export_cost_main')

# export_cost_constr=m.addConstrs((export_cost[r]==imp[r]*(export_mine.prod(mine_exp_cost)+export_proc.prod(proc_exp_cost)+export_cath.prod(cath_exp_cost)+export_cell.prod(cell_export_cost)+export_pack.prod(pack_export_cost)+export_rec.prod(rec_export_cost)) for r in R), name='export_cost')

In [ ]:
inventory_cost=m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'inventory_cost')
inventory_cost_constr=m.addConstrs((
    inventory_cost[r]==invR*(
    quicksum(stockpile_mine[y, 'mine', t, r]* mine_prod_cost[y,'mine', t, r] for y in Y for t in T_mine)+
    quicksum(stockpile_proc[y,'proc', i, o, r] * proc_prod_cost[y,'proc', i, o, r] for y in Y for i in T_mine for o in T_proc)+
    quicksum(stockpile_cath[y,'cath', i, o, r] * cath_prod_cost[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath)+
    quicksum(stockpile_cell[y,'cell', i, o, r] * cell_prod_cost[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell)+
    quicksum(stockpile_pack[y,'pack', i, o, r] * pack_prod_cost[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack)+
     quicksum(stockpile_rec[y,'rec', i, o, r] * rec_prod_cost[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec)   
    )
                       for r in R), name='inventory_cost')         

In [ ]:
total_cost = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'total_cost')
total_cost_constr = m.addConstrs((total_cost[r] == mine_cost[r]+ proc_cost[r]+ cath_cost[r]+ cell_cost[r]+ pack_cost[r]+import_cost[r]+rec_cost[r]+inventory_cost[r]+import_cost[r] for r in R),  name='total_cost_constr')

### Emissions Variables (add CO2 for import, export, stockpile)

In [ ]:
mine_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'mine_CO2')

mine_CO2_constr = m.addConstrs((
    mine_CO2 [r] == alpha[r] * (
        quicksum(prod_mine[y, 'mine', t, r]* mine_prod_CO2[y,'mine', t, r] for y in Y for t in T_mine) +
        quicksum(new_capacity_mine[y, 'mine',t, r] * mine_var_CO2[y,'mine',t, r] for y in Y for t in T_mine) +
        quicksum(invest_mine[y,'mine',t, r] * mine_fixed_CO2[y,'mine',t, r] for y in Y for t in T_mine)
    ) 
    for r in R
), name='mine_CO2_constr')  

#mine_CO2_constr = m.addConstrs((mine_CO2 [r]== prod_mine.prod(mine_prod_CO2) + new_capacity_mine.prod(mine_var_CO2) + invest_mine.prod(mine_fixed_CO2) for r in R),  name='mine_CO2_constr')

In [ ]:
proc_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'proc_CO2')

proc_CO2_constr = m.addConstrs((
    proc_CO2[r] == alpha[r] * (
        quicksum(prod_proc[y,'proc', i, o, r] * proc_prod_CO2[y,'proc', i, o, r] for y in Y for i in T_mine for o in T_proc) +
        quicksum(new_capacity_proc[y, 'proc', i, o, r] * proc_var_CO2[y, 'proc', i, o, r] for y in Y for i in T_mine for o in T_proc) +
        quicksum(invest_proc[y,'proc', i, o, r] * proc_fixed_CO2[y,'proc', i, o, r] for y in Y for i in T_mine for o in T_proc)
    )
    for r in R
), name='proc_CO2_constr')

#proc_CO2_constr = m.addConstrs((proc_CO2[r] == prod_proc.prod(proc_prod_CO2) + new_capacity_proc.prod(proc_var_CO2) + invest_proc.prod(proc_fixed_CO2) for r in R),  name='proc_CO2_constr')

In [ ]:
cath_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'cath_CO2')

cath_CO2_constr = m.addConstrs((
    cath_CO2[r] == alpha[r] * (
        quicksum(prod_cath[y,'cath', i, o, r] * cath_prod_CO2[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath) +
        quicksum(new_capacity_cath[y,'cath', i, o, r] * cath_var_CO2[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath) +
        quicksum(invest_cath[y,'cath', i, o, r] * cath_fixed_CO2[y,'cath', i, o, r] for y in Y for i in T_proc for o in T_cath)
    )
    for r in R
), name='cath_CO2_constr')

#cath_CO2_constr = m.addConstrs((cath_CO2[r] == prod_cath.prod(cath_prod_CO2) + new_capacity_cath.prod(cath_var_CO2) + invest_cath.prod(cath_fixed_CO2) for r in R),  name='cath_CO2_constr')

In [ ]:
cell_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'cell_CO2')

cell_CO2_constr = m.addConstrs((
    cell_CO2[r] == alpha[r] * (
        quicksum(prod_cell[y,'cell', i, o, r] * cell_prod_CO2[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell) +
        quicksum(new_capacity_cell[y,'cell', i, o, r] * cell_var_CO2[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell) +
        quicksum(invest_cell[y,'cell', i, o, r] * cell_fixed_CO2[y,'cell', i, o, r] for y in Y for i in T_cath for o in T_cell)
    )
    for r in R
), name='cell_CO2_constr')

#cell_CO2_constr = m.addConstrs((cell_CO2[r] == prod_cell.prod(cell_prod_CO2) + new_capacity_cell.prod(cell_var_CO2) + invest_cell.prod(cell_fixed_CO2) for r in R),  name='cell_CO2_constr')

In [ ]:
pack_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'pack_CO2')
pack_CO2_constr = m.addConstrs((
    pack_CO2[r] == alpha[r] * (
        quicksum(prod_pack[y,'pack', i, o, r] * pack_prod_CO2[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack) +
        quicksum(new_capacity_pack[y,'pack', i, o, r] * pack_var_CO2[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack) +
        quicksum(invest_pack[y,'pack', i, o, r] * pack_fixed_CO2[y,'pack', i, o, r] for y in Y for i in T_cell for o in T_pack)
    )
    for r in R
), name='pack_CO2_constr')

#pack_CO2_constr = m.addConstrs((pack_CO2[r] == prod_pack.prod(pack_prod_CO2) + new_capacity_pack.prod(pack_var_CO2) + invest_pack.prod(pack_fixed_CO2) for r in R),  name='pack_CO2_constr')

In [ ]:
rec_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'rec_CO2')
rec_CO2_constr = m.addConstrs((
    rec_CO2[r] == alpha[r] * (
        quicksum(prod_rec[y,'rec', i, o, r] * rec_prod_CO2[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec) +
        quicksum(new_capacity_rec[y,'rec', i, o, r] * rec_var_CO2[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec) +
        quicksum(invest_rec[y,'rec', i, o, r] * rec_fixed_CO2[y,'rec', i, o, r] for y in Y for i in T_pack for o in T_rec)
    )
    for r in R
), name='rec_CO2_constr')

#rec_CO2_constr = m.addConstrs((rec_CO2[r] == prod_rec.prod(rec_prod_CO2) + new_capacity_rec.prod(rec_var_CO2) + invest_rec.prod(rec_fixed_CO2) for r in R),  name='rec_CO2_constr')

In [ ]:
import_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'import_CO2')
import_cost_constr=m.addConstrs((
    import_CO2[r]==impCO2*(
        quicksum(import_mine[y,'mine', t, r, r_out]*mine_imp_CO2[y,'mine',t, r, r_out] for y in Y for t in T_mine for r_out in R if r_out != r)+
        quicksum(import_proc[y,'proc',i,o, r, r_out]*proc_imp_CO2[y,'proc',i,o, r, r_out] for y in Y for i in T_mine for o in T_proc for r_out in R if r_out != r)+
        quicksum(import_cath[y,'cath',i,o, r, r_out]*cath_imp_CO2[y,'cath',i,o, r, r_out] for y in Y for i in T_proc for o in T_cath for r_out in R if r_out != r)+
        quicksum(import_cell[y,'cell',i,o, r, r_out]*cell_import_CO2[y,'cell',i,o, r, r_out] for y in Y for i in T_cath for o in T_cell for r_out in R if r_out != r)+
        quicksum(import_pack[y,'pack',i,o, r, r_out]*pack_import_CO2[y,'pack',i,o, r, r_out] for y in Y for i in T_cell for o in T_pack for r_out in R if r_out != r)+
        quicksum(import_rec.sum(y,'rec','*',o, r, r_out)*rec_import_CO2.sum(y,'rec','*',o, r, r_out) for y in Y for o in T_rec for r_out in R if r_out != r)
                                )
                                 for r in R), name='import_CO2')
#import_CO2_constr = m.addConstrs((import_CO2[r] == impCO2*(import_mine.prod(mine_imp_CO2)+import_proc.prod(proc_imp_CO2)+import_cath.prod(cath_imp_CO2)+import_cell.prod(cell_import_CO2)+import_pack.prod(pack_import_CO2)+import_rec.prod(rec_import_CO2)) for r in R), name='import_cost')

In [ ]:
export_CO2 = m.addVars(R, lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'export_CO2')
export_CO2_constr=m.addConstrs((
    export_CO2[r]==imp[r]*(
        quicksum(export_mine[y,'mine', t, r_out, r]*mine_exp_CO2[y,'mine',t, r_out, r] for y in Y for t in T_mine for r_out in R if r_out != r)+
        quicksum(export_proc[y,'proc',i,o, r_out, r]*proc_exp_CO2[y,'proc',i,o, r_out, r] for y in Y for i in T_mine for o in T_proc for r_out in R if r_out != r)+
        quicksum(export_cath[y,'cath',i,o, r_out, r]*cath_exp_CO2[y,'cath',i,o, r_out,r] for y in Y for i in T_proc for o in T_cath for r_out in R if r_out != r)+
        quicksum(export_cell[y,'cell',i,o, r_out,r]*cell_export_CO2[y,'cell',i,o, r_out, r] for y in Y for i in T_cath for o in T_cell for r_out in R if r_out != r)+
        quicksum(export_pack[y,'pack',i,o, r_out, r]*pack_export_CO2[y,'pack',i,o, r_out, r] for y in Y for i in T_cell for o in T_pack for r_out in R if r_out != r)+
        quicksum(export_rec.sum(y,'rec','*',o, r_out,r)*rec_export_CO2.sum(y,'rec','*',o, r_out,r) for y in Y for o in T_rec for r_out in R if r_out != r)
                                )
                                 for r in R), name='export_CO2')
#export_CO2_constr = m.addConstrs((export_CO2[r] == impCO2*(export_mine.prod(mine_exp_CO2)+export_proc.prod(proc_exp_CO2)+export_cath.prod(cath_exp_CO2)+export_cell.prod(cell_export_CO2)+export_pack.prod(pack_export_CO2)+export_rec.prod(rec_export_CO2)) for r in R), name='export_cost')

In [ ]:
total_CO2 = m.addVar(lb = 0, ub = GRB.INFINITY, vtype = GRB.CONTINUOUS, name = 'total_CO2')
total_CO2_constr = m.addConstr(total_CO2 == mine_CO2.sum()+ proc_CO2.sum()+ cath_CO2.sum()+ cell_CO2.sum()+ pack_CO2.sum()+ rec_CO2.sum()+import_CO2.sum()+export_CO2.sum(),  name='total_CO2_constr')

### Objective Function

In [ ]:
m.setObjective(total_cost.sum(), GRB.MINIMIZE)
#

# Optimization Parameters and Function

## Writing the Model to a File

In [ ]:
# Save Model File
m.write('model.mps')
# Save Model File
m.write('model_SaraTrial77.lp')

## Calling the Optimization Function

In [ ]:
# TimeLimit in Seconds
m.Params.TimeLimit = 240
# Node Limit
# m.Params.NodeLimit = 10

In [ ]:
# Read Previous Solution
# m.read('solution.sol')
#Run optimization engine
m.optimize()

In [ ]:
# Check if the model is infeasible
if m.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    
    # Compute the Irreducible Inconsistent Subsystem (IIS)
    m.computeIIS()
    
    # Save the IIS to a file for detailed inspection (optional)
    m.write("model_infeasible.ilp")
    
    print("\nIIS computed. These are the constraints that make the model infeasible:")
    
    # Iterate through the constraints and check which ones are in the IIS
    for c in m.getConstrs():
        if c.IISConstr:  # If the constraint is part of the IIS
            print(f"Infeasible constraint: {c.ConstrName}")
else:
    print("Model is feasible.")

# Problem Solution


## Objective Value Solution

In [ ]:
#print(m.ObjVal)
m.ObjVal

## Variable Solution


In [ ]:
# Loop through each region and print the costs for that region

# Mine Cost by Region
for r in R:
    print(f"Mine Cost for {r}: $ (MM)", f"{mine_cost[r].X:,.0f}")

# Proc Cost by Region
for r in R:
    print(f"Proc Cost for {r}: $ (MM)", f"{proc_cost[r].X:,.0f}")

# Cath Cost by Region
for r in R:
    print(f"Cath Cost for {r}: $ (MM)", f"{cath_cost[r].X:,.0f}")

# Cell Cost by Region
for r in R:
    print(f"Cell Cost for {r}: $ (MM)", f"{cell_cost[r].X:,.0f}")

# Pack Cost by Region
for r in R:
    print(f"Pack Cost for {r}: $ (MM)", f"{pack_cost[r].X:,.0f}")

# Recycle Cost by Region
for r in R:
    print(f"Recycle Cost for {r}: $ (MM)", f"{rec_cost[r].X:,.0f}")
    
# Import Cost by Region
for r in R:
    print(f"Import Cost for {r}: $ (MM)", f"{import_cost[r].X:,.0f}")    
    
# Export Cost by Region
for r in R:
    print(f"Export Cost for {r}: $ (MM)", f"{export_cost_main[r].X:,.0f}")  
    
# Inventory Cost by Region
for r in R:
    print(f"Inventory Cost for {r}: $ (MM)", f"{inventory_cost[r].X:,.0f}")  
    
# Total Cost by Region
for r in R:
    region_total_cost = (mine_cost[r].X
                         + proc_cost[r].X
                         + cath_cost[r].X
                         + cell_cost[r].X
                         + pack_cost[r].X
                         + rec_cost[r].X
                        +import_cost[r].X
                        +inventory_cost[r].X)
    
    print(f"Total Cost for {r}: $ (MM)", f"{region_total_cost:,.0f}")

In [ ]:
# Total Mine CO2 Cost by Region
for r in R:
    total_mine_CO2 = mine_CO2[r].X
    print(f"Total Mine CO2 for {r}: $ (kt)", f"{total_mine_CO2:,.0f}")

# Total Proc CO2 Cost by Region
for r in R:
    total_proc_CO2 = proc_CO2[r].X
    print(f"Total Proc CO2 for {r}: $ (kt)", f"{total_proc_CO2:,.0f}")

# Total Cath CO2 Cost by Region
for r in R:
    total_cath_CO2 = cath_CO2[r].X
    print(f"Total Cath CO2 for {r}: $ (kt)", f"{total_cath_CO2:,.0f}")

# Total Cell CO2 Cost by Region
for r in R:
    total_cell_CO2 = cell_CO2[r].X
    print(f"Total Cell CO2 for {r}: $ (kt)", f"{total_cell_CO2:,.0f}")

# Total Pack CO2 Cost by Region
for r in R:
    total_pack_CO2 = pack_CO2[r].X
    print(f"Total Pack CO2 for {r}: $ (kt)", f"{total_pack_CO2:,.0f}")

# Total Recycle CO2 Cost by Region
for r in R:
    total_rec_CO2 = rec_CO2[r].X
    print(f"Total Recycle CO2 for {r}: $ (kt)", f"{total_rec_CO2:,.0f}")
    
# Total Import CO2 Cost by Region
for r in R:
    total_import_CO2 = import_CO2[r].X
    print(f"Total Import CO2 for {r}: $ (kt)", f"{total_import_CO2:,.0f}")    

# Overall Total CO2 Cost by Region
for r in R:
    total_CO2_cost = (mine_CO2[r].X + proc_CO2[r].X + cath_CO2[r].X + cell_CO2[r].X + pack_CO2[r].X + rec_CO2[r].X+import_CO2[r].X)
    print(f"Total CO2 Cost for {r}: $ (kt)", f"{total_CO2_cost:,.0f}")

In [ ]:
# Variable info

varInfo = [(v.varName, v.X, v.LB, v.UB) for v in m.getVars() ]

df = pd.DataFrame(varInfo)

df.columns=['Variable Name','Solution Value', 'LB','UB']

df.to_excel("marketshare-var.xlsx", index=False)


df_var = df.query('`Solution Value` > 0')

df_var

## Constraints Solutions

Shows what each constraints values are and the equations for them.

In [ ]:
# Constraint info

constrInfo = [(c.constrName, m.getRow(c),m.getRow(c).getValue(), c.Sense, c.RHS) for c in m.getConstrs()]

df = pd.DataFrame(constrInfo)

df.columns=['Constraint Name','Constraint equation', 'Value','Sense','RHS']

df.to_excel("marketshare-constr.xlsx", index=False)

df

## Writing the Solution to a File

In [ ]:
# Save Model Solution
m.write('solution.sol')

## Dispose of Model to Avoid Confusion

In [ ]:
# Getting Rid of Model for Reruns
#m.dispose()

## Reading a Model and Solution and Solving It

In [ ]:
# m = gp.read('model.mps', env=env)
# m.read('solution.sol')
# m.optimize()


### Comparing Starting and Final Solutions from Read File

In [ ]:
# start_soln = m.getAttr(GRB.Attr.Start)
# final_soln = m.getAttr(GRB.Attr.Xn)
# diff_btwn_soln = [a - b for a, b in zip(start_soln, final_soln)]
# diff_btwn_soln

In [ ]:
print('Code Chunk Run Complete')